## Global optimized landmarks over keyframes (from logged `meta_data.npz`)

This notebook:
- Loads `meta_data.npz` written by the Point2Pose pipeline (`save_meta_data: true`)
- Finds keyframes (`is_key_frame == True`)
- Replays a lightweight global landmark optimizer (LM graph) over keyframes to recover
  `key_points_optimized` + `key_points_idx_optimized` per keyframe update
- Plots an interactive 3D Plotly slider similar to `12a. frame_optimization_analysis_debug.ipynb`

### Usage

1. Set `META_DATA_NPZ` in the next cell.
2. Run all cells.


In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# --- User config ---
# Path to the logged NPZ (DataLogger output)
META_DATA_NPZ = "/home/justin/results/eccv_point2pose/final_results/ho3d_all_final/MPM10/meta_data/meta_data.npz"
# META_DATA_NPZ = "/home/justin/code/point-to-pose/results/ycbmultitrackreal/006_mustard_bottle_010_potted_meat_can_hard/meta_data/meta_data.npz"
# Dataset config for GT overlay / reconstruction
DATASET = "ho3d"  # "ho3d", "ycbinisaac", or "auto"
HO3D_ROOT = "/home/justin/data/HO3D_V3"
YCBINISAAC_ROOT = "/home/justin/data/YCBInIsaac"
YCBINISAAC_OBJECT_NAME = None  # Optional override, e.g. "cracker_box"

OBJ_ID = 0
MAX_KEYFRAMES = 600  # e.g. 40 to limit runtime

# Plot params
PLOT_WIDTH = 1200
PLOT_HEIGHT = 850
FIX_AXIS_RANGES = True
ASPECT_MODE = "cube"  # "data" or "cube"

# Replay params
# POSE_FIELD_PREFERENCE = ("pose_local", "pose_frontend", "obj_pose")
POSE_FIELD_PREFERENCE = ("obj_pose",)
SUPPRESS_OPTIMIZER_PRINTS = True

# Ensure repo root is importable (when running from the `notebook/` directory)
_cwd = os.path.abspath(os.getcwd())
_repo_root = os.path.abspath(os.path.join(_cwd, "..")) if os.path.basename(_cwd) == "notebook" else _cwd
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

assert os.path.exists(META_DATA_NPZ), f"META_DATA_NPZ not found: {META_DATA_NPZ}"


In [ ]:
import contextlib
import io


def _ragged_slice(npz, key: str, row_idx: int) -> np.ndarray:
    """Return the flattened ragged payload for one row."""
    data = npz[f"{key}_data"]
    offsets = npz[f"{key}_offsets"]
    lengths = npz[f"{key}_lengths"]

    off = int(offsets[row_idx])
    L = int(lengths[row_idx])
    if L <= 0:
        return np.asarray([], dtype=data.dtype)
    return np.asarray(data[off : off + L])


def _ragged_slice_2d(npz, key: str, row_idx: int, d: int, *, dtype=None) -> np.ndarray:
    """Return ragged payload reshaped as (-1, d)."""
    flat = _ragged_slice(npz, key, row_idx)
    if dtype is not None:
        flat = flat.astype(dtype, copy=False)
    if flat.size == 0:
        return np.zeros((0, d), dtype=(dtype or flat.dtype))
    if flat.size % d != 0:
        raise ValueError(f"Ragged field {key} row {row_idx}: flat.size={flat.size} not divisible by d={d}")
    return flat.reshape(-1, d)


def _choose_pose(meta, row_idx: int, prefer=POSE_FIELD_PREFERENCE) -> np.ndarray:
    """Pick a 4x4 pose matrix from the logged meta arrays."""
    for k in prefer:
        if k not in meta.files:
            continue
        p = meta[k][row_idx]
        if p is None:
            continue
        p = np.asarray(p, dtype=float)
        if p.shape == (4, 4):
            return p
    raise KeyError(f"None of pose fields present/valid at row {row_idx}: {prefer}")


In [ ]:
# Load logged meta data
meta = np.load(META_DATA_NPZ, allow_pickle=True)

if "frame_id" not in meta.files or "is_key_frame" not in meta.files:
    raise KeyError(
        "meta_data.npz is missing required keys. Expected at least: frame_id, is_key_frame. "
        f"Found keys: {sorted(meta.files)[:20]} ..."
    )

frame_ids = np.asarray(meta["frame_id"], dtype=int)
is_key_frame = np.asarray(meta["is_key_frame"], dtype=bool)

kf_rows = np.where(is_key_frame)[0].astype(int)
if MAX_KEYFRAMES is not None:
    kf_rows = kf_rows[: int(MAX_KEYFRAMES)]

print(f"Loaded meta rows: {len(frame_ids)}")
print(f"Keyframes in log:  {int(is_key_frame.sum())} (using {len(kf_rows)})")

# --- Build landmark snapshots directly from the logged pipeline state ---
#
# The pipeline logs `obj_key_points` *after* its per-frame work, including any
# keyframe graph updates. For frames where `is_key_frame == True`, we treat the
# logged `obj_key_points` as the "global optimized landmarks" snapshot.
#
# Landmark IDs here are the (stable) indices in the object's keypoint array.

if "obj_key_points_data" not in meta.files:
    raise KeyError(
        "meta_data.npz is missing obj_key_points (expected ragged fields: obj_key_points_data/offsets/lengths)."
    )

global_landmarks_updates = []

for kf_idx, row_idx in enumerate(kf_rows.tolist()):
    frame_id = int(frame_ids[row_idx])

    xyz = _ragged_slice_2d(meta, "obj_key_points", row_idx, 3, dtype=float)

    # Stable per-landmark ids: index in the object keypoint array
    ids = np.arange(xyz.shape[0], dtype=int)

    # Optionally filter invalid keypoints (keep original indices as ids)
    if "obj_valid_data" in meta.files:
        valid = _ragged_slice(meta, "obj_valid", row_idx).astype(bool, copy=False).reshape(-1)
        if valid.size == xyz.shape[0]:
            ids = np.flatnonzero(valid).astype(int)
            xyz = xyz[valid]

    m = np.isfinite(xyz).all(axis=1)
    xyz = xyz[m]
    ids = ids[m]

    global_landmarks_updates.append(
        {
            "obj_id": int(OBJ_ID),
            "kf_idx": int(kf_idx),
            "frame_id": int(frame_id),
            "xyz": xyz,
            "ids": ids,
        }
    )

print(f"Collected global_landmarks_updates: {len(global_landmarks_updates)}")
if len(global_landmarks_updates) > 0:
    print(
        "First/last kf_idx:",
        global_landmarks_updates[0]["kf_idx"],
        global_landmarks_updates[-1]["kf_idx"],
    )

In [ ]:
# --- Plotly 3D slider (same idea as 12a) ---
if "global_landmarks_updates" not in globals() or not isinstance(global_landmarks_updates, list):
    print("No global_landmarks_updates found. Run the replay cell above first.")
elif len(global_landmarks_updates) == 0:
    print("global_landmarks_updates is empty. (Likely only the first keyframe ran, or the optimizer returned no landmarks.)")
else:
    updates = [u for u in global_landmarks_updates if int(u.get("obj_id", -1)) == int(OBJ_ID)]
    if len(updates) == 0:
        print(f"No landmark snapshots found for OBJ_ID={OBJ_ID}.")
    else:
        # Sort by keyframe index to make the slider monotonic
        updates = sorted(updates, key=lambda u: int(u.get("kf_idx", -1)))

        try:
            import plotly.graph_objects as go
        except Exception as e:
            raise ImportError(
                "Plotly is required for the interactive 3D plot. Install with `pip install plotly` and re-run this cell."
            ) from e

        # Establish global color range over ids
        _all_ids = []
        _all_xyz = []
        for u in updates:
            ids = u.get("ids", None)
            xyz = u.get("xyz", None)
            if ids is not None:
                ids = np.asarray(ids, dtype=int).reshape(-1)
                if ids.size > 0:
                    _all_ids.append(ids)
            if xyz is not None:
                xyz = np.asarray(xyz, dtype=float)
                m = np.isfinite(xyz).all(axis=1)
                if np.any(m):
                    _all_xyz.append(xyz[m])

        if len(_all_ids) == 0:
            raise RuntimeError("No landmark ids found in global_landmarks_updates.")
        if len(_all_xyz) == 0:
            raise RuntimeError("No finite landmark xyz found in global_landmarks_updates.")

        all_ids = np.concatenate(_all_ids, axis=0)
        cmin = float(np.min(all_ids))
        cmax = float(np.max(all_ids))

        all_xyz = np.concatenate(_all_xyz, axis=0)
        xmin, ymin, zmin = np.min(all_xyz, axis=0)
        xmax, ymax, zmax = np.max(all_xyz, axis=0)

        # Fixed axis ranges (so switching frames won't auto-rescale)
        scene_axes = dict(
            xaxis_title="x",
            yaxis_title="y",
            zaxis_title="z",
            aspectmode=ASPECT_MODE,
        )
        if FIX_AXIS_RANGES:
            # Pad and optionally make cubic
            spans = np.array([xmax - xmin, ymax - ymin, zmax - zmin], dtype=float)
            spans = np.maximum(spans, 1e-6)
            pad = 0.05 * float(np.max(spans))

            cx, cy, cz = float(0.5 * (xmin + xmax)), float(0.5 * (ymin + ymax)), float(0.5 * (zmin + zmax))
            if ASPECT_MODE == "cube":
                half = 0.5 * float(np.max(spans)) + pad
                xr = [cx - half, cx + half]
                yr = [cy - half, cy + half]
                zr = [cz - half, cz + half]
            else:
                xr = [float(xmin - pad), float(xmax + pad)]
                yr = [float(ymin - pad), float(ymax + pad)]
                zr = [float(zmin - pad), float(zmax + pad)]

            scene_axes.update(
                dict(
                    xaxis=dict(range=xr, autorange=False),
                    yaxis=dict(range=yr, autorange=False),
                    zaxis=dict(range=zr, autorange=False),
                )
            )


        # Hide 3D axes and scene background
        scene_axes.update(
            dict(
                bgcolor="white",
                xaxis=dict(
                    **scene_axes.get("xaxis", {}),
                    visible=False,
                    showbackground=False,
                    showgrid=False,
                    zeroline=False,
                    showticklabels=False,
                    title="",
                ),
                yaxis=dict(
                    **scene_axes.get("yaxis", {}),
                    visible=False,
                    showbackground=False,
                    showgrid=False,
                    zeroline=False,
                    showticklabels=False,
                    title="",
                ),
                zaxis=dict(
                    **scene_axes.get("zaxis", {}),
                    visible=False,
                    showbackground=False,
                    showgrid=False,
                    zeroline=False,
                    showticklabels=False,
                    title="",
                ),
            )
        )

        # Baseline snapshot (first keyframe update)
        base = updates[0]
        xyz0 = np.asarray(base["xyz"], dtype=float)
        ids0 = np.asarray(base["ids"], dtype=int).reshape(-1)
        m0 = np.isfinite(xyz0).all(axis=1)

        fig = go.Figure()

        fig.add_trace(
            go.Scatter3d(
                x=xyz0[m0, 0],
                y=xyz0[m0, 1],
                z=xyz0[m0, 2],
                mode="markers",
                marker=dict(size=2, color="rgba(140,140,140,0.45)"),
                text=[f"id={int(i)}" for i in ids0[m0]],
                hovertemplate="%{text}<extra></extra>",
                name=f"baseline (kf_idx={int(base.get('kf_idx', -1))}, frame_id={int(base.get('frame_id', -1))})",
            )
        )

        # One trace per update; slider toggles visibility
        for k, u in enumerate(updates):
            xyz = np.asarray(u["xyz"], dtype=float)
            ids = np.asarray(u["ids"], dtype=int).reshape(-1)
            m = np.isfinite(xyz).all(axis=1)
            kf_idx = int(u.get("kf_idx", -1))
            frame_id = int(u.get("frame_id", -1))

            fig.add_trace(
                go.Scatter3d(
                    x=xyz[m, 0],
                    y=xyz[m, 1],
                    z=xyz[m, 2],
                    mode="markers",
                    marker=dict(
                        size=4,
                        color=ids[m].astype(float),
                        colorscale="Turbo",
                        cmin=cmin,
                        cmax=cmax,
                        opacity=0.9,
                        colorbar=dict(title="landmark id"),
                    ),
                    text=[f"id={int(i)}" for i in ids[m]],
                    hovertemplate="x=%{x:.4f}<br>y=%{y:.4f}<br>z=%{z:.4f}<br>%{text}<extra></extra>",
                    name=f"kf_idx={kf_idx}, frame_id={frame_id}",
                    visible=(k == 0),
                )
            )

        steps = []
        for k, u in enumerate(updates):
            vis = [True] + [False] * len(updates)  # baseline always on
            vis[1 + k] = True
            kf_idx = int(u.get("kf_idx", -1))
            frame_id = int(u.get("frame_id", -1))
            steps.append(
                dict(
                    method="update",
                    args=[
                        {"visible": vis},
                        {
                            "title": f"Global optimized landmarks over keyframes (obj {OBJ_ID}) — kf_idx={kf_idx}, frame_id={frame_id}",
                        },
                    ],
                    label=str(frame_id if frame_id >= 0 else kf_idx),
                )
            )

        fig.update_layout(
            title=f"Global optimized landmarks over keyframes (obj {OBJ_ID}) — kf_idx={int(updates[0].get('kf_idx', -1))}, frame_id={int(updates[0].get('frame_id', -1))}",
            margin=dict(l=0, r=0, b=0, t=55),
            width=PLOT_WIDTH,
            height=PLOT_HEIGHT,
            paper_bgcolor="white",
            plot_bgcolor="white",
            scene=scene_axes,
            sliders=[
                dict(
                    active=0,
                    currentvalue={"prefix": "frame_id: "},
                    steps=steps,
                )
            ],
            legend=dict(x=0.01, y=0.99),
            # Keeps user-driven camera/zoom across slider steps
            uirevision=f"kf_landmarks_obj_{OBJ_ID}",
        )

        fig.show()




In [ ]:
# --- Plotly 3D slider with observed points and correspondences ---
if "global_landmarks_updates" not in globals() or not isinstance(global_landmarks_updates, list):
    print("No global_landmarks_updates found. Run the replay cell above first.")
elif len(global_landmarks_updates) == 0:
    print("global_landmarks_updates is empty. (Likely only the first keyframe ran, or the optimizer returned no landmarks.)")
else:
    updates = [u for u in global_landmarks_updates if int(u.get("obj_id", -1)) == int(OBJ_ID)]
    if len(updates) == 0:
        print(f"No landmark snapshots found for OBJ_ID={OBJ_ID}.")
    else:
        # Sort by keyframe index to make the slider monotonic
        updates = sorted(updates, key=lambda u: int(u.get("kf_idx", -1)))

        try:
            import plotly.graph_objects as go
        except Exception as e:
            raise ImportError(
                "Plotly is required for the interactive 3D plot. Install with `pip install plotly` and re-run this cell."
            ) from e

        # Import transform utilities
        from point2pose.utils.transform import transform_pts, inverse_SE3

        # Load observed points and correspondences for each keyframe
        keyframe_data = []
        for kf_idx, row_idx in enumerate(kf_rows.tolist()):
            frame_id = int(frame_ids[row_idx])
            
            # Get object pose for this keyframe (to transform camera frame to object frame)
            # Follow the same pattern as notebook 12. frame_optimization_analysis.ipynb
            obj_pose = None
            if 'pose_frontend' in meta.files:
                obj_pose = meta['pose_frontend'][row_idx]
            elif 'pose_local' in meta.files:
                obj_pose = meta['pose_local'][row_idx]
            elif 'obj_pose' in meta.files:
                obj_pose = meta['obj_pose'][row_idx]
            elif 'obj_init_pose' in meta.files:
                obj_pose = meta['obj_init_pose'][row_idx]
            
            if obj_pose is not None:
                obj_pose = np.asarray(obj_pose, dtype=float)
                if obj_pose.shape == (4, 4):
                    # obj_pose transforms from object frame to camera frame
                    # We need inverse to transform camera frame to object frame
                    T_c2o = inverse_SE3(obj_pose)
                else:
                    T_c2o = None
            else:
                # If pose not available, skip transformation (will show incorrect positions)
                T_c2o = None
            
            
            # Get global optimized landmarks (map points)
            xyz_map = _ragged_slice_2d(meta, "obj_key_points", row_idx, 3, dtype=float)
            ids_map = np.arange(xyz_map.shape[0], dtype=int)
            
            # Filter valid keypoints
            if "obj_valid_data" in meta.files:
                valid = _ragged_slice(meta, "obj_valid", row_idx).astype(bool, copy=False).reshape(-1)
                if valid.size == xyz_map.shape[0]:
                    ids_map = np.flatnonzero(valid).astype(int)
                    xyz_map = xyz_map[valid]
            
            m = np.isfinite(xyz_map).all(axis=1)
            xyz_map = xyz_map[m]
            ids_map = ids_map[m]
            
            # Get observed points (reg_curr3d) and corresponding map points (reg_key_points)
            # Note: reg_curr3d is in camera frame, reg_key_points is in object frame
            xyz_observed = None
            xyz_correspond = None
            if "reg_curr3d_data" in meta.files and "reg_key_points_data" in meta.files:
                xyz_observed_cam = _ragged_slice_2d(meta, "reg_curr3d", row_idx, 3, dtype=float)
                xyz_correspond = _ragged_slice_2d(meta, "reg_key_points", row_idx, 3, dtype=float)
                
                # Transform observed points from camera frame to object frame
                if T_c2o is not None and xyz_observed_cam.shape[0] > 0:
                    xyz_observed = transform_pts(T_c2o, xyz_observed_cam)
                else:
                    xyz_observed = xyz_observed_cam.copy() if xyz_observed_cam.shape[0] > 0 else np.zeros((0, 3), dtype=float)
                
                # Filter finite points - only if both arrays have the same number of points
                if xyz_observed.shape[0] == xyz_correspond.shape[0] and xyz_observed.shape[0] > 0:
                    m_obs = np.isfinite(xyz_observed).all(axis=1) & np.isfinite(xyz_correspond).all(axis=1)
                    if np.any(m_obs):
                        xyz_observed = xyz_observed[m_obs]
                        xyz_correspond = xyz_correspond[m_obs]
                    else:
                        xyz_observed = np.zeros((0, 3), dtype=float)
                        xyz_correspond = np.zeros((0, 3), dtype=float)
                else:
                    # Mismatched sizes or empty arrays - filter each separately but don't create correspondences
                    if xyz_observed.shape[0] > 0:
                        m_obs = np.isfinite(xyz_observed).all(axis=1)
                        xyz_observed = xyz_observed[m_obs] if np.any(m_obs) else np.zeros((0, 3), dtype=float)
                    else:
                        xyz_observed = np.zeros((0, 3), dtype=float)
                    # Set correspond to empty since sizes don't match
                    xyz_correspond = np.zeros((0, 3), dtype=float)
            
            # Get residuals for correspondence error coloring
            residuals = None
            if "reg_residuals_data" in meta.files and xyz_observed is not None and xyz_correspond is not None:
                residuals_flat = _ragged_slice(meta, "reg_residuals", row_idx)
                if residuals_flat.size > 0:
                    residuals = np.asarray(residuals_flat, dtype=float)
                    # Filter to match the filtered observed/correspond points
                    # We need to apply the same filtering that was applied to observed/correspond points
                    if "reg_curr3d_data" in meta.files and "reg_key_points_data" in meta.files:
                        xyz_observed_cam_raw = _ragged_slice_2d(meta, "reg_curr3d", row_idx, 3, dtype=float)
                        xyz_correspond_raw = _ragged_slice_2d(meta, "reg_key_points", row_idx, 3, dtype=float)
                        
                        if residuals.size == xyz_observed_cam_raw.shape[0] and xyz_observed_cam_raw.shape[0] == xyz_correspond_raw.shape[0]:
                            # Apply same filtering as observed points
                            m_obs = np.isfinite(xyz_observed_cam_raw).all(axis=1) & np.isfinite(xyz_correspond_raw).all(axis=1)
                            if np.any(m_obs):
                                residuals = residuals[m_obs]
                            else:
                                residuals = np.zeros((0,), dtype=float)
                        else:
                            residuals = np.zeros((0,), dtype=float)
                    else:
                        residuals = np.zeros((0,), dtype=float)
                else:
                    residuals = np.zeros((0,), dtype=float)
            else:
                residuals = np.zeros((0,), dtype=float)
            
            keyframe_data.append({
                "kf_idx": int(kf_idx),
                "frame_id": int(frame_id),
                "xyz_map": xyz_map,
                "ids_map": ids_map,
                "xyz_observed": xyz_observed if xyz_observed is not None else np.zeros((0, 3), dtype=float),
                "xyz_correspond": xyz_correspond if xyz_correspond is not None else np.zeros((0, 3), dtype=float),
                "residuals": residuals if residuals is not None else np.zeros((0,), dtype=float),
            })

        # Establish axis ranges (no longer need color range since using fixed colors)
        _all_xyz = []
        for kfd in keyframe_data:
            xyz = kfd.get("xyz_map", None)
            if xyz is not None and xyz.size > 0:
                m = np.isfinite(xyz).all(axis=1)
                if np.any(m):
                    _all_xyz.append(xyz[m])
            
            # Also include observed points for axis range
            xyz_obs = kfd.get("xyz_observed", None)
            if xyz_obs is not None and xyz_obs.size > 0:
                m = np.isfinite(xyz_obs).all(axis=1)
                if np.any(m):
                    _all_xyz.append(xyz_obs[m])

        if len(_all_xyz) == 0:
            raise RuntimeError("No finite landmark xyz found in keyframe_data.")
        
        # Error threshold for correspondence coloring (adjust as needed)
        ERROR_THRESHOLD = 0.01  # meters

        all_xyz = np.concatenate(_all_xyz, axis=0)
        xmin, ymin, zmin = np.min(all_xyz, axis=0)
        xmax, ymax, zmax = np.max(all_xyz, axis=0)

        # Fixed axis ranges
        scene_axes = dict(
            xaxis_title="x",
            yaxis_title="y",
            zaxis_title="z",
            aspectmode=ASPECT_MODE,
        )
        if FIX_AXIS_RANGES:
            spans = np.array([xmax - xmin, ymax - ymin, zmax - zmin], dtype=float)
            spans = np.maximum(spans, 1e-6)
            pad = 0.05 * float(np.max(spans))

            cx, cy, cz = float(0.5 * (xmin + xmax)), float(0.5 * (ymin + ymax)), float(0.5 * (zmin + zmax))
            if ASPECT_MODE == "cube":
                half = 0.5 * float(np.max(spans)) + pad
                xr = [cx - half, cx + half]
                yr = [cy - half, cy + half]
                zr = [cz - half, cz + half]
            else:
                xr = [float(xmin - pad), float(xmax + pad)]
                yr = [float(ymin - pad), float(ymax + pad)]
                zr = [float(zmin - pad), float(zmax + pad)]

            scene_axes.update(
                dict(
                    xaxis=dict(range=xr, autorange=False),
                    yaxis=dict(range=yr, autorange=False),
                    zaxis=dict(range=zr, autorange=False),
                )
            )


        # Hide 3D axes and scene background
        scene_axes.update(
            dict(
                bgcolor="white",
                xaxis=dict(
                    **scene_axes.get("xaxis", {}),
                    visible=False,
                    showbackground=False,
                    showgrid=False,
                    zeroline=False,
                    showticklabels=False,
                    title="",
                ),
                yaxis=dict(
                    **scene_axes.get("yaxis", {}),
                    visible=False,
                    showbackground=False,
                    showgrid=False,
                    zeroline=False,
                    showticklabels=False,
                    title="",
                ),
                zaxis=dict(
                    **scene_axes.get("zaxis", {}),
                    visible=False,
                    showbackground=False,
                    showgrid=False,
                    zeroline=False,
                    showticklabels=False,
                    title="",
                ),
            )
        )

        # Baseline snapshot (first keyframe)
        base = keyframe_data[0]
        xyz0 = np.asarray(base["xyz_map"], dtype=float)
        ids0 = np.asarray(base["ids_map"], dtype=int).reshape(-1)
        m0 = np.isfinite(xyz0).all(axis=1)

        fig = go.Figure()

        # Baseline map (always visible) - same pretty blue color
        MAP_COLOR_BASELINE = "rgba(70,130,180,0.45)"  # Steel blue, more transparent for baseline
        fig.add_trace(
            go.Scatter3d(
                x=xyz0[m0, 0],
                y=xyz0[m0, 1],
                z=xyz0[m0, 2],
                mode="markers",
                marker=dict(size=2, color=MAP_COLOR_BASELINE),
                text=[f"id={int(i)}" for i in ids0[m0]],
                hovertemplate="%{text}<extra></extra>",
                name=f"baseline map (kf_idx={int(base.get('kf_idx', -1))}, frame_id={int(base.get('frame_id', -1))})",
            )
        )

        # For each keyframe, add:
        # 1. Map points (key points) for this keyframe
        # 2. Observed points
        # 3. Correspondence lines
        for k, kfd in enumerate(keyframe_data):
            kf_idx = int(kfd.get("kf_idx", -1))
            frame_id = int(kfd.get("frame_id", -1))
            
            # Map points (key points) for this keyframe - pretty blue color
            xyz_map = np.asarray(kfd["xyz_map"], dtype=float)
            ids_map = np.asarray(kfd["ids_map"], dtype=int).reshape(-1)
            m_map = np.isfinite(xyz_map).all(axis=1)
            
            # Pretty blue color for map points
            MAP_COLOR = "rgba(70,130,180,0.85)"  # Steel blue
            
            fig.add_trace(
                go.Scatter3d(
                    x=xyz_map[m_map, 0],
                    y=xyz_map[m_map, 1],
                    z=xyz_map[m_map, 2],
                    mode="markers",
                    marker=dict(
                        size=6,
                        color=MAP_COLOR,
                        opacity=0.9,
                    ),
                    text=[f"id={int(i)}, frame={frame_id}" for i in ids_map[m_map]],
                    hovertemplate="x=%{x:.4f}<br>y=%{y:.4f}<br>z=%{z:.4f}<br>%{text}<extra></extra>",
                    name=f"key points (kf_idx={kf_idx}, frame_id={frame_id})",
                    visible=(k == 0),
                )
            )
            
            # Observed points (already filtered when loaded) - pretty coral/salmon color
            xyz_observed = np.asarray(kfd["xyz_observed"], dtype=float)
            
            # Pretty coral/salmon color for observed points
            OBSERVED_COLOR = "rgba(255,127,80,0.85)"  # Coral
            
            if xyz_observed.size > 0:
                fig.add_trace(
                    go.Scatter3d(
                        x=xyz_observed[:, 0],
                        y=xyz_observed[:, 1],
                        z=xyz_observed[:, 2],
                        mode="markers",
                        marker=dict(
                            size=6,
                            color=OBSERVED_COLOR,
                            # symbol="circle" is default, so we don't need to specify
                        ),
                        text=[f"obs_{i}" for i in range(xyz_observed.shape[0])],
                        hovertemplate="x=%{x:.4f}<br>y=%{y:.4f}<br>z=%{z:.4f}<br>%{text}<extra></extra>",
                        name=f"observed points (kf_idx={kf_idx}, frame_id={frame_id})",
                        visible=(k == 0),
                    )
                )
            else:
                # Add empty trace to maintain index consistency
                fig.add_trace(
                    go.Scatter3d(
                        x=[],
                        y=[],
                        z=[],
                        mode="markers",
                        marker=dict(size=6, color=OBSERVED_COLOR),
                        name=f"observed points (kf_idx={kf_idx}, frame_id={frame_id})",
                        visible=(k == 0),
                    )
                )
            
            # Correspondence lines from observed points to corresponding map points
            # Color by error: red if above threshold, green otherwise
            xyz_correspond = np.asarray(kfd["xyz_correspond"], dtype=float)
            residuals = np.asarray(kfd.get("residuals", np.zeros((0,), dtype=float)), dtype=float)
            
            if xyz_observed.size > 0 and xyz_correspond.size > 0 and xyz_observed.shape[0] == xyz_correspond.shape[0]:
                # Prettier colors: darker green and red
                GREEN_COLOR = "rgba(34,139,34,0.6)"  # Forest green
                RED_COLOR = "rgba(178,34,34,0.6)"    # Firebrick red
                
                # Create line segments for each correspondence with color based on error
                x_lines_green = []
                y_lines_green = []
                z_lines_green = []
                x_lines_red = []
                y_lines_red = []
                z_lines_red = []
                
                for i in range(xyz_observed.shape[0]):
                    if residuals.size > i and residuals[i] > ERROR_THRESHOLD:
                        # High error - red
                        x_lines_red.extend([xyz_observed[i, 0], xyz_correspond[i, 0], None])
                        y_lines_red.extend([xyz_observed[i, 1], xyz_correspond[i, 1], None])
                        z_lines_red.extend([xyz_observed[i, 2], xyz_correspond[i, 2], None])
                    else:
                        # Low error - green
                        x_lines_green.extend([xyz_observed[i, 0], xyz_correspond[i, 0], None])
                        y_lines_green.extend([xyz_observed[i, 1], xyz_correspond[i, 1], None])
                        z_lines_green.extend([xyz_observed[i, 2], xyz_correspond[i, 2], None])
                
                # Add green lines (low error)
                if len(x_lines_green) > 0:
                    fig.add_trace(
                        go.Scatter3d(
                            x=x_lines_green,
                            y=y_lines_green,
                            z=z_lines_green,
                            mode="lines",
                            line=dict(color=GREEN_COLOR, width=4),
                            name=f"correspondences (low error) (kf_idx={kf_idx}, frame_id={frame_id})",
                            visible=(k == 0),
                            showlegend=True,
                        )
                    )
                else:
                    # Add empty trace to maintain index consistency
                    fig.add_trace(
                        go.Scatter3d(
                            x=[],
                            y=[],
                            z=[],
                            mode="lines",
                            line=dict(color=GREEN_COLOR, width=4),
                            name=f"correspondences (low error) (kf_idx={kf_idx}, frame_id={frame_id})",
                            visible=(k == 0),
                            showlegend=True,
                        )
                    )
                
                # Add red lines (high error)
                if len(x_lines_red) > 0:
                    fig.add_trace(
                        go.Scatter3d(
                            x=x_lines_red,
                            y=y_lines_red,
                            z=z_lines_red,
                            mode="lines",
                            line=dict(color=RED_COLOR, width=4),
                            name=f"correspondences (high error) (kf_idx={kf_idx}, frame_id={frame_id})",
                            visible=(k == 0),
                            showlegend=True,
                        )
                    )
                else:
                    # Add empty trace to maintain index consistency
                    fig.add_trace(
                        go.Scatter3d(
                            x=[],
                            y=[],
                            z=[],
                            mode="lines",
                            line=dict(color=RED_COLOR, width=4),
                            name=f"correspondences (high error) (kf_idx={kf_idx}, frame_id={frame_id})",
                            visible=(k == 0),
                            showlegend=True,
                        )
                    )
            else:
                # Add empty traces to maintain index consistency
                GREEN_COLOR = "rgba(34,139,34,0.6)"
                RED_COLOR = "rgba(178,34,34,0.6)"
                fig.add_trace(
                    go.Scatter3d(
                        x=[],
                        y=[],
                        z=[],
                        mode="lines",
                        line=dict(color=GREEN_COLOR, width=4),
                        name=f"correspondences (low error) (kf_idx={kf_idx}, frame_id={frame_id})",
                        visible=(k == 0),
                        showlegend=True,
                    )
                )
                fig.add_trace(
                    go.Scatter3d(
                        x=[],
                        y=[],
                        z=[],
                        mode="lines",
                        line=dict(color=RED_COLOR, width=4),
                        name=f"correspondences (high error) (kf_idx={kf_idx}, frame_id={frame_id})",
                        visible=(k == 0),
                        showlegend=True,
                    )
                )

        # Create slider steps
        # Each keyframe has 4 traces: map points, observed points, green correspondences, red correspondences
        # Plus 1 baseline trace
        steps = []
        for k, kfd in enumerate(keyframe_data):
            kf_idx = int(kfd.get("kf_idx", -1))
            frame_id = int(kfd.get("frame_id", -1))
            
            # Visibility: baseline always on, then 4 traces per keyframe
            vis = [True]  # baseline
            for i in range(len(keyframe_data)):
                # Map points
                vis.append(i == k)
                # Observed points
                vis.append(i == k)
                # Green correspondences (low error)
                vis.append(i == k)
                # Red correspondences (high error)
                vis.append(i == k)
            
            steps.append(
                dict(
                    method="update",
                    args=[
                        {"visible": vis},
                        {
                            "title": f"Map with observed points and correspondences (obj {OBJ_ID}) — kf_idx={kf_idx}, frame_id={frame_id}",
                        },
                    ],
                    label=str(frame_id if frame_id >= 0 else kf_idx),
                )
            )

        fig.update_layout(
            title=f"Map with observed points and correspondences (obj {OBJ_ID}) — kf_idx={int(keyframe_data[0].get('kf_idx', -1))}, frame_id={int(keyframe_data[0].get('frame_id', -1))}",
            margin=dict(l=0, r=0, b=0, t=55),
            width=PLOT_WIDTH,
            height=PLOT_HEIGHT,
            paper_bgcolor="white",
            plot_bgcolor="white",
            scene=scene_axes,
            sliders=[
                dict(
                    active=0,
                    currentvalue={"prefix": "frame_id: "},
                    steps=steps,
                )
            ],
            legend=dict(x=0.01, y=0.99),
            uirevision=f"kf_obs_corr_obj_{OBJ_ID}",
        )

        fig.show()



In [ ]:
# --- Plotly 3D slider with observed points, correspondences, AND newly sampled keypoints ---
if "global_landmarks_updates" not in globals() or not isinstance(global_landmarks_updates, list):
    print("No global_landmarks_updates found. Run the replay cell above first.")
elif len(global_landmarks_updates) == 0:
    print("global_landmarks_updates is empty. (Likely only the first keyframe ran, or the optimizer returned no landmarks.)")
else:
    updates = [u for u in global_landmarks_updates if int(u.get("obj_id", -1)) == int(OBJ_ID)]
    if len(updates) == 0:
        print(f"No landmark snapshots found for OBJ_ID={OBJ_ID}.")
    else:
        # Sort by keyframe index to make the slider monotonic
        updates = sorted(updates, key=lambda u: int(u.get("kf_idx", -1)))

        try:
            import plotly.graph_objects as go
        except Exception as e:
            raise ImportError(
                "Plotly is required for the interactive 3D plot. Install with `pip install plotly` and re-run this cell."
            ) from e

        # Import transform utilities
        from point2pose.utils.transform import transform_pts, inverse_SE3

        # Load observed points, correspondences, and newly sampled keypoints for each keyframe
        keyframe_data = []
        for kf_idx, row_idx in enumerate(kf_rows.tolist()):
            frame_id = int(frame_ids[row_idx])
            
            # Get object pose for this keyframe (to transform camera frame to object frame)
            # Follow the same pattern as notebook 12. frame_optimization_analysis.ipynb
            obj_pose = None
            if 'pose_frontend' in meta.files:
                obj_pose = meta['pose_frontend'][row_idx]
            elif 'pose_local' in meta.files:
                obj_pose = meta['pose_local'][row_idx]
            elif 'obj_pose' in meta.files:
                obj_pose = meta['obj_pose'][row_idx]
            elif 'obj_init_pose' in meta.files:
                obj_pose = meta['obj_init_pose'][row_idx]
            
            if obj_pose is not None:
                obj_pose = np.asarray(obj_pose, dtype=float)
                if obj_pose.shape == (4, 4):
                    # obj_pose transforms from object frame to camera frame
                    # We need inverse to transform camera frame to object frame
                    T_c2o = inverse_SE3(obj_pose)
                else:
                    T_c2o = None
            else:
                # If pose not available, skip transformation (will show incorrect positions)
                T_c2o = None
            
            
            # Get global optimized landmarks (map points)
            xyz_map = _ragged_slice_2d(meta, "obj_key_points", row_idx, 3, dtype=float)
            ids_map = np.arange(xyz_map.shape[0], dtype=int)
            
            # Filter valid keypoints
            if "obj_valid_data" in meta.files:
                valid = _ragged_slice(meta, "obj_valid", row_idx).astype(bool, copy=False).reshape(-1)
                if valid.size == xyz_map.shape[0]:
                    ids_map = np.flatnonzero(valid).astype(int)
                    xyz_map = xyz_map[valid]
            
            m = np.isfinite(xyz_map).all(axis=1)
            xyz_map = xyz_map[m]
            ids_map = ids_map[m]
            
            # Get newly sampled keypoints for this keyframe (keypoints added in this frame)
            xyz_newly_sampled = None
            ids_newly_sampled = None
            if "obj_key_point_frames_data" in meta.files:
                key_point_frames = _ragged_slice(meta, "obj_key_point_frames", row_idx).astype(int, copy=False)
                if key_point_frames.size == xyz_map.shape[0]:
                    # Find keypoints that were sampled in this frame
                    newly_sampled_mask = (key_point_frames == frame_id)
                    if np.any(newly_sampled_mask):
                        xyz_newly_sampled = xyz_map[newly_sampled_mask]
                        ids_newly_sampled = ids_map[newly_sampled_mask]
                    else:
                        xyz_newly_sampled = np.zeros((0, 3), dtype=float)
                        ids_newly_sampled = np.zeros((0,), dtype=int)
                else:
                    xyz_newly_sampled = np.zeros((0, 3), dtype=float)
                    ids_newly_sampled = np.zeros((0,), dtype=int)
            else:
                xyz_newly_sampled = np.zeros((0, 3), dtype=float)
                ids_newly_sampled = np.zeros((0,), dtype=int)
            
            # Get observed points (reg_curr3d) and corresponding map points (reg_key_points)
            # Note: reg_curr3d is in camera frame, reg_key_points is in object frame
            xyz_observed = None
            xyz_correspond = None
            if "reg_curr3d_data" in meta.files and "reg_key_points_data" in meta.files:
                xyz_observed_cam = _ragged_slice_2d(meta, "reg_curr3d", row_idx, 3, dtype=float)
                xyz_correspond = _ragged_slice_2d(meta, "reg_key_points", row_idx, 3, dtype=float)
                
                # Transform observed points from camera frame to object frame
                if T_c2o is not None and xyz_observed_cam.shape[0] > 0:
                    xyz_observed = transform_pts(T_c2o, xyz_observed_cam)
                else:
                    xyz_observed = xyz_observed_cam.copy() if xyz_observed_cam.shape[0] > 0 else np.zeros((0, 3), dtype=float)
                
                # Filter finite points - only if both arrays have the same number of points
                if xyz_observed.shape[0] == xyz_correspond.shape[0] and xyz_observed.shape[0] > 0:
                    m_obs = np.isfinite(xyz_observed).all(axis=1) & np.isfinite(xyz_correspond).all(axis=1)
                    if np.any(m_obs):
                        xyz_observed = xyz_observed[m_obs]
                        xyz_correspond = xyz_correspond[m_obs]
                    else:
                        xyz_observed = np.zeros((0, 3), dtype=float)
                        xyz_correspond = np.zeros((0, 3), dtype=float)
                else:
                    # Mismatched sizes or empty arrays - filter each separately but don't create correspondences
                    if xyz_observed.shape[0] > 0:
                        m_obs = np.isfinite(xyz_observed).all(axis=1)
                        xyz_observed = xyz_observed[m_obs] if np.any(m_obs) else np.zeros((0, 3), dtype=float)
                    else:
                        xyz_observed = np.zeros((0, 3), dtype=float)
                    # Set correspond to empty since sizes don't match
                    xyz_correspond = np.zeros((0, 3), dtype=float)
            
            # Get residuals for correspondence error coloring
            residuals = None
            if "reg_residuals_data" in meta.files and xyz_observed is not None and xyz_correspond is not None:
                residuals_flat = _ragged_slice(meta, "reg_residuals", row_idx)
                if residuals_flat.size > 0:
                    residuals = np.asarray(residuals_flat, dtype=float)
                    # Filter to match the filtered observed/correspond points
                    # We need to apply the same filtering that was applied to observed/correspond points
                    if "reg_curr3d_data" in meta.files and "reg_key_points_data" in meta.files:
                        xyz_observed_cam_raw = _ragged_slice_2d(meta, "reg_curr3d", row_idx, 3, dtype=float)
                        xyz_correspond_raw = _ragged_slice_2d(meta, "reg_key_points", row_idx, 3, dtype=float)
                        
                        if residuals.size == xyz_observed_cam_raw.shape[0] and xyz_observed_cam_raw.shape[0] == xyz_correspond_raw.shape[0]:
                            # Apply same filtering as observed points
                            m_obs = np.isfinite(xyz_observed_cam_raw).all(axis=1) & np.isfinite(xyz_correspond_raw).all(axis=1)
                            if np.any(m_obs):
                                residuals = residuals[m_obs]
                            else:
                                residuals = np.zeros((0,), dtype=float)
                        else:
                            residuals = np.zeros((0,), dtype=float)
                    else:
                        residuals = np.zeros((0,), dtype=float)
                else:
                    residuals = np.zeros((0,), dtype=float)
            else:
                residuals = np.zeros((0,), dtype=float)
            
            keyframe_data.append({
                "kf_idx": int(kf_idx),
                "frame_id": int(frame_id),
                "xyz_map": xyz_map,
                "ids_map": ids_map,
                "xyz_newly_sampled": xyz_newly_sampled if xyz_newly_sampled is not None else np.zeros((0, 3), dtype=float),
                "ids_newly_sampled": ids_newly_sampled if ids_newly_sampled is not None else np.zeros((0,), dtype=int),
                "xyz_observed": xyz_observed if xyz_observed is not None else np.zeros((0, 3), dtype=float),
                "xyz_correspond": xyz_correspond if xyz_correspond is not None else np.zeros((0, 3), dtype=float),
                "residuals": residuals if residuals is not None else np.zeros((0,), dtype=float),
            })

        # Establish axis ranges (no longer need color range since using fixed colors)
        _all_xyz = []
        for kfd in keyframe_data:
            xyz = kfd.get("xyz_map", None)
            if xyz is not None and xyz.size > 0:
                m = np.isfinite(xyz).all(axis=1)
                if np.any(m):
                    _all_xyz.append(xyz[m])
            
            # Also include observed points and newly sampled points for axis range
            xyz_obs = kfd.get("xyz_observed", None)
            if xyz_obs is not None and xyz_obs.size > 0:
                m = np.isfinite(xyz_obs).all(axis=1)
                if np.any(m):
                    _all_xyz.append(xyz_obs[m])
            
            xyz_new = kfd.get("xyz_newly_sampled", None)
            if xyz_new is not None and xyz_new.size > 0:
                m = np.isfinite(xyz_new).all(axis=1)
                if np.any(m):
                    _all_xyz.append(xyz_new[m])

        if len(_all_xyz) == 0:
            raise RuntimeError("No finite landmark xyz found in keyframe_data.")
        
        # Error threshold for correspondence coloring (adjust as needed)
        ERROR_THRESHOLD = 0.01  # meters

        all_xyz = np.concatenate(_all_xyz, axis=0)
        xmin, ymin, zmin = np.min(all_xyz, axis=0)
        xmax, ymax, zmax = np.max(all_xyz, axis=0)

        # Fixed axis ranges
        scene_axes = dict(
            xaxis_title="x",
            yaxis_title="y",
            zaxis_title="z",
            aspectmode=ASPECT_MODE,
        )
        if FIX_AXIS_RANGES:
            spans = np.array([xmax - xmin, ymax - ymin, zmax - zmin], dtype=float)
            spans = np.maximum(spans, 1e-6)
            pad = 0.05 * float(np.max(spans))

            cx, cy, cz = float(0.5 * (xmin + xmax)), float(0.5 * (ymin + ymax)), float(0.5 * (zmin + zmax))
            if ASPECT_MODE == "cube":
                half = 0.5 * float(np.max(spans)) + pad
                xr = [cx - half, cx + half]
                yr = [cy - half, cy + half]
                zr = [cz - half, cz + half]
            else:
                xr = [float(xmin - pad), float(xmax + pad)]
                yr = [float(ymin - pad), float(ymax + pad)]
                zr = [float(zmin - pad), float(zmax + pad)]

            scene_axes.update(
                dict(
                    xaxis=dict(range=xr, autorange=False),
                    yaxis=dict(range=yr, autorange=False),
                    zaxis=dict(range=zr, autorange=False),
                )
            )


        # Hide 3D axes and scene background
        scene_axes.update(
            dict(
                bgcolor="white",
                xaxis=dict(
                    **scene_axes.get("xaxis", {}),
                    visible=False,
                    showbackground=False,
                    showgrid=False,
                    zeroline=False,
                    showticklabels=False,
                    title="",
                ),
                yaxis=dict(
                    **scene_axes.get("yaxis", {}),
                    visible=False,
                    showbackground=False,
                    showgrid=False,
                    zeroline=False,
                    showticklabels=False,
                    title="",
                ),
                zaxis=dict(
                    **scene_axes.get("zaxis", {}),
                    visible=False,
                    showbackground=False,
                    showgrid=False,
                    zeroline=False,
                    showticklabels=False,
                    title="",
                ),
            )
        )

        # Baseline snapshot (first keyframe)
        base = keyframe_data[0]
        xyz0 = np.asarray(base["xyz_map"], dtype=float)
        ids0 = np.asarray(base["ids_map"], dtype=int).reshape(-1)
        m0 = np.isfinite(xyz0).all(axis=1)

        fig = go.Figure()

        # Baseline map (always visible) - same pretty blue color
        MAP_COLOR_BASELINE = "rgba(70,130,180,0.45)"  # Steel blue, more transparent for baseline
        fig.add_trace(
            go.Scatter3d(
                x=xyz0[m0, 0],
                y=xyz0[m0, 1],
                z=xyz0[m0, 2],
                mode="markers",
                marker=dict(size=2, color=MAP_COLOR_BASELINE),
                text=[f"id={int(i)}" for i in ids0[m0]],
                hovertemplate="%{text}<extra></extra>",
                name=f"baseline map (kf_idx={int(base.get('kf_idx', -1))}, frame_id={int(base.get('frame_id', -1))})",
            )
        )

        # For each keyframe, add:
        # 1. Map points (key points) for this keyframe
        # 2. Newly sampled keypoints for this keyframe
        # 3. Observed points
        # 4. Correspondence lines
        for k, kfd in enumerate(keyframe_data):
            kf_idx = int(kfd.get("kf_idx", -1))
            frame_id = int(kfd.get("frame_id", -1))
            
            # Map points (key points) for this keyframe - pretty blue color
            xyz_map = np.asarray(kfd["xyz_map"], dtype=float)
            ids_map = np.asarray(kfd["ids_map"], dtype=int).reshape(-1)
            m_map = np.isfinite(xyz_map).all(axis=1)
            
            # Pretty blue color for map points
            MAP_COLOR = "rgba(70,130,180,0.85)"  # Steel blue
            
            fig.add_trace(
                go.Scatter3d(
                    x=xyz_map[m_map, 0],
                    y=xyz_map[m_map, 1],
                    z=xyz_map[m_map, 2],
                    mode="markers",
                    marker=dict(
                        size=4,
                        color=MAP_COLOR,
                        opacity=0.9,
                    ),
                    text=[f"id={int(i)}, frame={frame_id}" for i in ids_map[m_map]],
                    hovertemplate="x=%{x:.4f}<br>y=%{y:.4f}<br>z=%{z:.4f}<br>%{text}<extra></extra>",
                    name=f"key points (kf_idx={kf_idx}, frame_id={frame_id})",
                    visible=(k == 0),
                )
            )
            
            # Newly sampled keypoints for this keyframe - pretty purple/magenta color
            xyz_newly_sampled = np.asarray(kfd["xyz_newly_sampled"], dtype=float)
            ids_newly_sampled = np.asarray(kfd["ids_newly_sampled"], dtype=int).reshape(-1)
            
            # Pretty purple/magenta color for newly sampled keypoints
            NEWLY_SAMPLED_COLOR = "rgba(186,85,211,0.9)"  # Medium orchid
            
            if xyz_newly_sampled.size > 0:
                m_new = np.isfinite(xyz_newly_sampled).all(axis=1)
                if np.any(m_new):
                    fig.add_trace(
                        go.Scatter3d(
                            x=xyz_newly_sampled[m_new, 0],
                            y=xyz_newly_sampled[m_new, 1],
                            z=xyz_newly_sampled[m_new, 2],
                            mode="markers",
                            marker=dict(
                                size=6,
                                color=NEWLY_SAMPLED_COLOR,
                                # symbol="circle" is default, so we don't need to specify
                            ),
                            text=[f"new_kp_id={int(i)}, frame={frame_id}" for i in ids_newly_sampled[m_new]],
                            hovertemplate="x=%{x:.4f}<br>y=%{y:.4f}<br>z=%{z:.4f}<br>%{text}<extra></extra>",
                            name=f"newly sampled keypoints (kf_idx={kf_idx}, frame_id={frame_id})",
                            visible=(k == 0),
                        )
                    )
                else:
                    # Add empty trace to maintain index consistency
                    fig.add_trace(
                        go.Scatter3d(
                            x=[],
                            y=[],
                            z=[],
                            mode="markers",
                            marker=dict(size=6, color=NEWLY_SAMPLED_COLOR),
                            name=f"newly sampled keypoints (kf_idx={kf_idx}, frame_id={frame_id})",
                            visible=(k == 0),
                        )
                    )
            else:
                # Add empty trace to maintain index consistency
                fig.add_trace(
                    go.Scatter3d(
                        x=[],
                        y=[],
                        z=[],
                        mode="markers",
                        marker=dict(size=6, color=NEWLY_SAMPLED_COLOR),
                        name=f"newly sampled keypoints (kf_idx={kf_idx}, frame_id={frame_id})",
                        visible=(k == 0),
                    )
                )
            
            # Observed points (already filtered when loaded) - pretty coral/salmon color
            xyz_observed = np.asarray(kfd["xyz_observed"], dtype=float)
            
            # Pretty coral/salmon color for observed points
            OBSERVED_COLOR = "rgba(255,127,80,0.85)"  # Coral
            
            if xyz_observed.size > 0:
                fig.add_trace(
                    go.Scatter3d(
                        x=xyz_observed[:, 0],
                        y=xyz_observed[:, 1],
                        z=xyz_observed[:, 2],
                        mode="markers",
                        marker=dict(
                            size=5,
                            color=OBSERVED_COLOR,
                            # symbol="circle" is default, so we don't need to specify
                        ),
                        text=[f"obs_{i}" for i in range(xyz_observed.shape[0])],
                        hovertemplate="x=%{x:.4f}<br>y=%{y:.4f}<br>z=%{z:.4f}<br>%{text}<extra></extra>",
                        name=f"observed points (kf_idx={kf_idx}, frame_id={frame_id})",
                        visible=(k == 0),
                    )
                )
            else:
                # Add empty trace to maintain index consistency
                fig.add_trace(
                    go.Scatter3d(
                        x=[],
                        y=[],
                        z=[],
                        mode="markers",
                        marker=dict(size=5, color=OBSERVED_COLOR),
                        name=f"observed points (kf_idx={kf_idx}, frame_id={frame_id})",
                        visible=(k == 0),
                    )
                )
            
            # Correspondence lines from observed points to corresponding map points
            # Color by error: red if above threshold, green otherwise
            xyz_correspond = np.asarray(kfd["xyz_correspond"], dtype=float)
            residuals = np.asarray(kfd.get("residuals", np.zeros((0,), dtype=float)), dtype=float)
            
            # Also show the correspond points explicitly so we can see where lines connect
            if xyz_correspond.size > 0:
                m_corr = np.isfinite(xyz_correspond).all(axis=1)
                if np.any(m_corr):
                    # Show correspond points with a distinct color (yellow/gold)
                    CORRESPOND_COLOR = "rgba(255,215,0,0.7)"  # Gold
                    fig.add_trace(
                        go.Scatter3d(
                            x=xyz_correspond[m_corr, 0],
                            y=xyz_correspond[m_corr, 1],
                            z=xyz_correspond[m_corr, 2],
                            mode="markers",
                            marker=dict(
                                size=4,
                                color=CORRESPOND_COLOR,
                                # symbol="circle" is default
                            ),
                            text=[f"corr_{i}" for i in range(np.sum(m_corr))],
                            hovertemplate="x=%{x:.4f}<br>y=%{y:.4f}<br>z=%{z:.4f}<br>%{text}<extra></extra>",
                            name=f"correspond points (kf_idx={kf_idx}, frame_id={frame_id})",
                            visible=(k == 0),
                        )
                    )
                else:
                    # Add empty trace to maintain index consistency
                    fig.add_trace(
                        go.Scatter3d(
                            x=[],
                            y=[],
                            z=[],
                            mode="markers",
                            marker=dict(size=4, color="rgba(255,215,0,0.7)"),
                            name=f"correspond points (kf_idx={kf_idx}, frame_id={frame_id})",
                            visible=(k == 0),
                        )
                    )
            else:
                # Add empty trace to maintain index consistency
                fig.add_trace(
                    go.Scatter3d(
                        x=[],
                        y=[],
                        z=[],
                        mode="markers",
                        marker=dict(size=4, color="rgba(255,215,0,0.7)"),
                        name=f"correspond points (kf_idx={kf_idx}, frame_id={frame_id})",
                        visible=(k == 0),
                    )
                )
            
            if xyz_observed.size > 0 and xyz_correspond.size > 0 and xyz_observed.shape[0] == xyz_correspond.shape[0]:
                # Prettier colors: darker green and red
                GREEN_COLOR = "rgba(34,139,34,0.6)"  # Forest green
                RED_COLOR = "rgba(178,34,34,0.6)"    # Firebrick red
                
                # Create line segments for each correspondence with color based on error
                x_lines_green = []
                y_lines_green = []
                z_lines_green = []
                x_lines_red = []
                y_lines_red = []
                z_lines_red = []
                
                for i in range(xyz_observed.shape[0]):
                    if residuals.size > i and residuals[i] > ERROR_THRESHOLD:
                        # High error - red
                        x_lines_red.extend([xyz_observed[i, 0], xyz_correspond[i, 0], None])
                        y_lines_red.extend([xyz_observed[i, 1], xyz_correspond[i, 1], None])
                        z_lines_red.extend([xyz_observed[i, 2], xyz_correspond[i, 2], None])
                    else:
                        # Low error - green
                        x_lines_green.extend([xyz_observed[i, 0], xyz_correspond[i, 0], None])
                        y_lines_green.extend([xyz_observed[i, 1], xyz_correspond[i, 1], None])
                        z_lines_green.extend([xyz_observed[i, 2], xyz_correspond[i, 2], None])
                
                # Add green lines (low error)
                if len(x_lines_green) > 0:
                    fig.add_trace(
                        go.Scatter3d(
                            x=x_lines_green,
                            y=y_lines_green,
                            z=z_lines_green,
                            mode="lines",
                            line=dict(color=GREEN_COLOR, width=4),
                            name=f"correspondences (low error) (kf_idx={kf_idx}, frame_id={frame_id})",
                            visible=(k == 0),
                            showlegend=True,
                        )
                    )
                else:
                    # Add empty trace to maintain index consistency
                    fig.add_trace(
                        go.Scatter3d(
                            x=[],
                            y=[],
                            z=[],
                            mode="lines",
                            line=dict(color=GREEN_COLOR, width=4),
                            name=f"correspondences (low error) (kf_idx={kf_idx}, frame_id={frame_id})",
                            visible=(k == 0),
                            showlegend=True,
                        )
                    )
                
                # Add red lines (high error)
                if len(x_lines_red) > 0:
                    fig.add_trace(
                        go.Scatter3d(
                            x=x_lines_red,
                            y=y_lines_red,
                            z=z_lines_red,
                            mode="lines",
                            line=dict(color=RED_COLOR, width=4),
                            name=f"correspondences (high error) (kf_idx={kf_idx}, frame_id={frame_id})",
                            visible=(k == 0),
                            showlegend=True,
                        )
                    )
                else:
                    # Add empty trace to maintain index consistency
                    fig.add_trace(
                        go.Scatter3d(
                            x=[],
                            y=[],
                            z=[],
                            mode="lines",
                            line=dict(color=RED_COLOR, width=4),
                            name=f"correspondences (high error) (kf_idx={kf_idx}, frame_id={frame_id})",
                            visible=(k == 0),
                            showlegend=True,
                        )
                    )
            else:
                # Add empty traces to maintain index consistency
                GREEN_COLOR = "rgba(34,139,34,0.6)"
                RED_COLOR = "rgba(178,34,34,0.6)"
                fig.add_trace(
                    go.Scatter3d(
                        x=[],
                        y=[],
                        z=[],
                        mode="lines",
                        line=dict(color=GREEN_COLOR, width=4),
                        name=f"correspondences (low error) (kf_idx={kf_idx}, frame_id={frame_id})",
                        visible=(k == 0),
                        showlegend=True,
                    )
                )
                fig.add_trace(
                    go.Scatter3d(
                        x=[],
                        y=[],
                        z=[],
                        mode="lines",
                        line=dict(color=RED_COLOR, width=4),
                        name=f"correspondences (high error) (kf_idx={kf_idx}, frame_id={frame_id})",
                        visible=(k == 0),
                        showlegend=True,
                    )
                )

        # Create slider steps
        # Each keyframe has 6 traces: map points, newly sampled keypoints, observed points, correspond points, green correspondences, red correspondences
        # Plus 1 baseline trace
        steps = []
        for k, kfd in enumerate(keyframe_data):
            kf_idx = int(kfd.get("kf_idx", -1))
            frame_id = int(kfd.get("frame_id", -1))
            
            # Visibility: baseline always on, then 6 traces per keyframe
            vis = [True]  # baseline
            for i in range(len(keyframe_data)):
                # Map points
                vis.append(i == k)
                # Newly sampled keypoints
                vis.append(i == k)
                # Observed points
                vis.append(i == k)
                # Correspond points
                vis.append(i == k)
                # Green correspondences (low error)
                vis.append(i == k)
                # Red correspondences (high error)
                vis.append(i == k)
            
            steps.append(
                dict(
                    method="update",
                    args=[
                        {"visible": vis},
                        {
                            "title": f"Map with observed points, correspondences, and newly sampled keypoints (obj {OBJ_ID}) — kf_idx={kf_idx}, frame_id={frame_id}",
                        },
                    ],
                    label=str(frame_id if frame_id >= 0 else kf_idx),
                )
            )

        fig.update_layout(
            title=f"Map with observed points, correspondences, and newly sampled keypoints (obj {OBJ_ID}) — kf_idx={int(keyframe_data[0].get('kf_idx', -1))}, frame_id={int(keyframe_data[0].get('frame_id', -1))}",
            margin=dict(l=0, r=0, b=0, t=55),
            width=PLOT_WIDTH,
            height=PLOT_HEIGHT,
            paper_bgcolor="white",
            plot_bgcolor="white",
            scene=scene_axes,
            sliders=[
                dict(
                    active=0,
                    currentvalue={"prefix": "frame_id: "},
                    steps=steps,
                )
            ],
            legend=dict(x=0.01, y=0.99),
            uirevision=f"kf_obs_corr_newkp_obj_{OBJ_ID}",
        )

        fig.show()



In [ ]:
# Build per-frame reg/landmark arrays and GT-based reconstructed keypoints (self-contained for this notebook)

from point2pose.io.sources.dataset.datareader import Ho3dReader, YCBInIsaacReader
from point2pose.utils.transform import inverse_SE3

# Map logged rows -> dense per-frame lists indexed by frame_id
max_fid = int(frame_ids.max())
num_frames = max_fid + 1

obj_key_points_list = [np.zeros((0, 3), dtype=float) for _ in range(num_frames)]
obj_key_point_frames_list = [np.zeros((0,), dtype=int) for _ in range(num_frames)]
est_pose_list = [None for _ in range(num_frames)]
is_key_frame_by_fid = np.zeros((num_frames,), dtype=bool)

for row_idx in range(len(frame_ids)):
    fid = int(frame_ids[row_idx])
    if fid < 0 or fid > max_fid:
        continue

    # Object keypoints and their source frames
    if "obj_key_points_data" in meta.files:
        obj_key_points_list[fid] = _ragged_slice_2d(meta, "obj_key_points", row_idx, 3, dtype=float)
    if "obj_key_point_frames_data" in meta.files:
        obj_key_point_frames_list[fid] = _ragged_slice(meta, "obj_key_point_frames", row_idx).astype(int, copy=False).reshape(-1)

    # Estimated pose (4x4) if available
    if "obj_pose" in meta.files:
        est_pose = meta["obj_pose"][row_idx]
        if est_pose is not None:
            est_pose = np.asarray(est_pose, dtype=float)
            if est_pose.shape == (4, 4):
                est_pose_list[fid] = est_pose

    # is_key_frame flag per frame id
    if is_key_frame[row_idx]:
        is_key_frame_by_fid[fid] = True

# Derive keyframe ids similar to notebook 13
key_frames = sorted({0} | set(np.where(is_key_frame_by_fid)[0].tolist()))
print(f"Total frames: {num_frames}, key_frames: {len(key_frames)} (first 20: {key_frames[:20]})")

# Infer video_name from META_DATA_NPZ path (e.g., .../results/ho3d_single/MPM10/meta_data/meta_data.npz)
_video_dir = os.path.dirname(META_DATA_NPZ)  # .../MPM10/meta_data
_video_parent = os.path.dirname(_video_dir)  # .../MPM10
video_name = os.path.basename(_video_parent)


def _is_valid_sequence_dir(path: str, *, require_cam_k: bool) -> bool:
    if path is None:
        return False
    if not os.path.isdir(path):
        return False
    if not os.path.isdir(os.path.join(path, "rgb")):
        return False
    if require_cam_k and not os.path.isfile(os.path.join(path, "cam_K.txt")):
        return False
    return True


def _resolve_dataset_name(dataset_hint: str) -> str:
    hint = str(dataset_hint).strip().lower()
    if hint in ("ho3d", "ycbinisaac"):
        return hint
    if hint != "auto":
        raise ValueError(f"Unsupported DATASET={dataset_hint!r}. Use 'ho3d', 'ycbinisaac', or 'auto'.")

    meta_path_lower = META_DATA_NPZ.lower()
    if "ycbinisaac" in meta_path_lower:
        return "ycbinisaac"
    return "ho3d"


def _resolve_video_dir(dataset_name: str, video_name: str):
    if dataset_name == "ho3d":
        roots = []
        if HO3D_ROOT is not None:
            roots.append(os.path.normpath(HO3D_ROOT))
            if os.path.basename(os.path.normpath(HO3D_ROOT)).lower() == "evaluation":
                roots.append(os.path.dirname(os.path.normpath(HO3D_ROOT)))

        for root in roots:
            for candidate in (
                os.path.join(root, "evaluation", video_name),
                os.path.join(root, video_name),
            ):
                if _is_valid_sequence_dir(candidate, require_cam_k=False):
                    return candidate
        return None

    if dataset_name == "ycbinisaac":
        roots = []
        if YCBINISAAC_ROOT is not None:
            roots.append(os.path.normpath(YCBINISAAC_ROOT))

        for root in roots:
            for candidate in (
                root,
                os.path.join(root, video_name),
                os.path.join(root, video_name, video_name),
            ):
                if _is_valid_sequence_dir(candidate, require_cam_k=True):
                    return candidate
        return None

    raise ValueError(f"Unsupported dataset: {dataset_name}")


resolved_dataset = _resolve_dataset_name(DATASET)
print(f"Resolved dataset={resolved_dataset}, video_name={video_name}")

# Configure/infer reader
reader = None
reader_obj_name = None
video_dir = _resolve_video_dir(resolved_dataset, video_name)

if video_dir is not None:
    try:
        if resolved_dataset == "ho3d":
            ho3d_root_for_reader = os.path.normpath(HO3D_ROOT)
            if os.path.basename(ho3d_root_for_reader).lower() == "evaluation":
                ho3d_root_for_reader = os.path.dirname(ho3d_root_for_reader)

            print(
                f"Creating Ho3dReader(video_dir={video_dir}, ho3d_root={ho3d_root_for_reader})"
            )
            reader = Ho3dReader(video_dir, ho3d_root_for_reader)
            print(f"Ho3dReader created with {len(reader)} frames")

        elif resolved_dataset == "ycbinisaac":
            print(f"Creating YCBInIsaacReader(video_dir={video_dir})")
            reader = YCBInIsaacReader(video_dir)
            print(f"YCBInIsaacReader created with {len(reader)} frames")

            object_names = reader.get_object_names()
            if YCBINISAAC_OBJECT_NAME is not None:
                if YCBINISAAC_OBJECT_NAME in object_names:
                    reader_obj_name = YCBINISAAC_OBJECT_NAME
                else:
                    print(
                        f"Warning: YCBINISAAC_OBJECT_NAME={YCBINISAAC_OBJECT_NAME!r} not found in {object_names}; falling back to OBJ_ID={OBJ_ID}."
                    )

            if reader_obj_name is None and len(object_names) > 0:
                if 0 <= int(OBJ_ID) < len(object_names):
                    reader_obj_name = object_names[int(OBJ_ID)]
                else:
                    reader_obj_name = object_names[0]
                    print(
                        f"Warning: OBJ_ID={OBJ_ID} out of range for object names {object_names}; using first object {reader_obj_name!r}."
                    )

            print(
                f"Using ycbinisaac object for GT pose queries: {reader_obj_name!r} (OBJ_ID={OBJ_ID})"
            )

    except Exception as e:
        print(f"Warning: could not create dataset reader for {resolved_dataset}: {e}")
        reader = None
else:
    if resolved_dataset == "ho3d":
        print(
            f"Warning: could not infer valid HO3D video_dir from HO3D_ROOT={HO3D_ROOT} and video_name={video_name}"
        )
    elif resolved_dataset == "ycbinisaac":
        print(
            f"Warning: could not infer valid ycbinisaac video_dir from YCBINISAAC_ROOT={YCBINISAAC_ROOT} and video_name={video_name}"
        )


def _reader_get_gt_pose(frame_id: int):
    if reader is None:
        return None
    if resolved_dataset == "ycbinisaac":
        return reader.get_gt_pose(frame_id, obj_name=reader_obj_name)
    return reader.get_gt_pose(frame_id)


# Build GT poses for keyframes and choose reference frame (first frame with GT)
gt_poses = {}
reference_frame_id = None
reference_gt_pose = None

if reader is not None:
    for kf_id in key_frames:
        if 0 <= kf_id < len(reader):
            pose = _reader_get_gt_pose(kf_id)
            if pose is not None:
                gt_poses[kf_id] = np.asarray(pose, dtype=float)

    # Choose reference frame: 0 if available, else first frame with GT
    if 0 in gt_poses:
        reference_frame_id = 0
        reference_gt_pose = gt_poses[0]
    else:
        for fid in range(len(reader)):
            pose = _reader_get_gt_pose(fid)
            if pose is not None:
                reference_frame_id = fid
                reference_gt_pose = np.asarray(pose, dtype=float)
                break

if reference_gt_pose is None:
    print("Warning: no GT reference frame found; GT-based reconstruction will be unavailable.")
else:
    print(f"Using frame {reference_frame_id} as GT reference frame.")

# Reconstruct all keypoints into the reference frame using GT poses when available
all_reconstructed_points = []
all_reconstructed_points_fid = []

if reference_gt_pose is not None:
    for kf_id in key_frames:
        if kf_id >= num_frames:
            continue

        kp_xyz = obj_key_points_list[kf_id]
        kp_frames = obj_key_point_frames_list[kf_id]
        if kp_xyz.size == 0 or kp_frames.size == 0:
            continue

        # Points that were initialized in this keyframe
        mask_init_here = kp_frames == kf_id
        if not np.any(mask_init_here):
            continue

        kp_xyz_kf = kp_xyz[mask_init_here]
        est_pose = est_pose_list[kf_id]
        if est_pose is None:
            continue

        if kf_id in gt_poses:
            # GT-aligned transform: ref <- GT_ref * inv(GT_kf) * est_pose
            T_ref_kf = reference_gt_pose @ inverse_SE3(gt_poses[kf_id]) @ est_pose
        else:
            # Fallback: keep in local frame (identity) if no GT
            T_ref_kf = np.eye(4, dtype=float)

        kp_h = np.hstack([kp_xyz_kf, np.ones((kp_xyz_kf.shape[0], 1), dtype=float)])
        kp_ref = (T_ref_kf @ kp_h.T).T[:, :3]

        all_reconstructed_points.append(kp_ref)
        all_reconstructed_points_fid.extend([kf_id] * kp_ref.shape[0])

if len(all_reconstructed_points) > 0:
    all_reconstructed_points = np.vstack(all_reconstructed_points)
    all_reconstructed_points_fid = np.asarray(all_reconstructed_points_fid, dtype=int)
    print(f"Reconstructed {all_reconstructed_points.shape[0]} GT-based keypoints across {len(key_frames)} keyframes.")
else:
    all_reconstructed_points = np.zeros((0, 3), dtype=float)
    all_reconstructed_points_fid = np.zeros((0,), dtype=int)
    print("No reconstructed GT-based keypoints (check GT availability and logged fields).")


### Estimated vs GT reconstructed keypoints (frame slider)
This block overlays **all** collected 3D keypoints built with the **estimated poses** vs. the same keypoints reconstructed using **GT poses** (see notebook 13).
Use the slider to pick a frame; optionally plot in the selected frame's coordinate system.

In [ ]:
# --- Interactive 3D: all keypoints (estimated-map vs GT-reconstructed) w.r.t selected frame ---

# Requirements: this cell expects you already ran the previous cells that define:
#   - obj_key_points_list (list of per-frame keypoints in the map / reference frame)
#   - est_pose_list (list of per-frame 4x4 poses, may include None)
#   - all_reconstructed_points, all_reconstructed_points_fid (GT-based reconstructed keypoints + their init keyframe id)
#   - reference_frame_id, reference_gt_pose, gt_poses, reader, _reader_get_gt_pose from the GT reconstruction cell

try:
    import plotly.graph_objects as go
except Exception as e:
    raise ImportError(
        "Plotly is required for this interactive 3D plot. Install with `pip install plotly` and re-run this cell."
    ) from e

import ipywidgets as widgets
from IPython.display import display

from point2pose.utils.transform import transform_pts, inverse_SE3

# ---------------------- Helpers ---------------------- #
def _finite_rows(xyz: np.ndarray) -> np.ndarray:
    if xyz is None:
        return np.zeros((0, 3), dtype=float)
    xyz = np.asarray(xyz, dtype=float).reshape(-1, 3)
    m = np.isfinite(xyz).all(axis=1)
    return xyz[m]


def _axis_ranges(xyz_a: np.ndarray, xyz_b: np.ndarray, pad_ratio: float = 0.05):
    xyz = np.vstack([xyz_a, xyz_b]) if (xyz_a.size and xyz_b.size) else (xyz_a if xyz_a.size else xyz_b)
    if xyz is None or xyz.size == 0:
        return None
    mn = xyz.min(axis=0)
    mx = xyz.max(axis=0)
    span = np.maximum(1e-6, mx - mn)
    pad = span * float(pad_ratio)
    mn = mn - pad
    mx = mx + pad
    return (mn[0], mx[0]), (mn[1], mx[1]), (mn[2], mx[2])


# ---------------------- Frame range ---------------------- #
if obj_key_points_list is None or len(obj_key_points_list) == 0:
    raise RuntimeError("obj_key_points_list is empty. Run the data-build cell above first.")

max_frame_id = len(obj_key_points_list) - 1

frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=max_frame_id,
    step=1,
    description="Frame",
    continuous_update=False,
)

# Toggle: plot in selected frame coordinate (recommended) vs reference/map coordinate
plot_in_selected_frame = widgets.Checkbox(
    value=True,
    description="Plot in selected frame coords",
)

# Only keep GT points that truly have GT available (optional but usually desired)
if "all_reconstructed_points_fid" in globals() and all_reconstructed_points_fid is not None:
    if "gt_poses" in globals() and isinstance(gt_poses, dict) and len(gt_poses) > 0:
        _gt_kf_ids = np.asarray(sorted(gt_poses.keys()), dtype=int)
        gt_has_gt_mask = np.isin(all_reconstructed_points_fid, _gt_kf_ids)
    else:
        gt_has_gt_mask = np.zeros_like(all_reconstructed_points_fid, dtype=bool)
else:
    gt_has_gt_mask = None


# ---------------------- Initial data ---------------------- #
def _get_points_for_frame(fid: int):
    # Estimated map snapshot at this frame (in reference/map coords)
    xyz_est = _finite_rows(obj_key_points_list[fid])

    # GT reconstructed points accumulated up to this frame (in reference/map coords)
    if all_reconstructed_points is None or len(all_reconstructed_points) == 0:
        xyz_gt = np.zeros((0, 3), dtype=float)
    else:
        m = all_reconstructed_points_fid <= fid
        if gt_has_gt_mask is not None:
            m = m & gt_has_gt_mask
        xyz_gt = _finite_rows(all_reconstructed_points[m])

    # Optionally re-express both sets in the selected frame coordinate system
    if plot_in_selected_frame.value:
        # Estimated: use relative transform fid <- reference_frame_id based on estimated poses (if available)
        if (
            reference_frame_id is not None
            and fid < len(est_pose_list)
            and reference_frame_id < len(est_pose_list)
            and est_pose_list[fid] is not None
            and est_pose_list[reference_frame_id] is not None
        ):
            try:
                T_f_ref_est = np.asarray(est_pose_list[fid], dtype=float) @ inverse_SE3(
                    np.asarray(est_pose_list[reference_frame_id], dtype=float)
                )
                xyz_est = transform_pts(T_f_ref_est, xyz_est)
            except Exception:
                pass  # fall back to reference/map coords

        # GT: use relative transform fid <- reference_frame_id based on GT poses (if available)
        if reader is not None and reference_gt_pose is not None:
            try:
                gt_pose_f = _reader_get_gt_pose(fid)
                if gt_pose_f is not None:
                    T_f_ref_gt = np.asarray(gt_pose_f, dtype=float) @ inverse_SE3(
                        np.asarray(reference_gt_pose, dtype=float)
                    )
                    xyz_gt = transform_pts(T_f_ref_gt, xyz_gt)
            except Exception:
                pass

    return xyz_est, xyz_gt


xyz_est0, xyz_gt0 = _get_points_for_frame(frame_slider.value)

# ---------------------- FigureWidget ---------------------- #
fig = go.FigureWidget(
    data=[
        go.Scatter3d(
            x=xyz_est0[:, 0],
            y=xyz_est0[:, 1],
            z=xyz_est0[:, 2],
            mode="markers",
            marker=dict(size=2, opacity=0.75, color="rgba(31,119,180,0.9)"),
            name="Estimated-map keypoints",
        ),
        go.Scatter3d(
            x=xyz_gt0[:, 0],
            y=xyz_gt0[:, 1],
            z=xyz_gt0[:, 2],
            mode="markers",
            marker=dict(size=2, opacity=0.85, color="rgba(255,127,14,0.9)"),
            name="GT-reconstructed keypoints",
        ),
    ]
)

fig.update_layout(
    width=PLOT_WIDTH if "PLOT_WIDTH" in globals() else 1200,
    height=PLOT_HEIGHT if "PLOT_HEIGHT" in globals() else 850,
    title=f"All keypoints - est vs GT (frame={frame_slider.value})",
    showlegend=True,
    paper_bgcolor="white",
    plot_bgcolor="white",
    scene=dict(
        aspectmode=ASPECT_MODE if "ASPECT_MODE" in globals() else "cube",
        bgcolor="white",
        xaxis=dict(visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
        yaxis=dict(visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
        zaxis=dict(visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
    ),
)

# Optional fixed axis ranges
if "FIX_AXIS_RANGES" in globals() and FIX_AXIS_RANGES:
    ranges = _axis_ranges(xyz_est0, xyz_gt0)
    if ranges is not None:
        (xr, yr, zr) = ranges
        fig.update_layout(
            scene=dict(
                xaxis=dict(range=list(xr), visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
                yaxis=dict(range=list(yr), visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
                zaxis=dict(range=list(zr), visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
                aspectmode=ASPECT_MODE if "ASPECT_MODE" in globals() else "cube",
                bgcolor="white",
            )
        )


# ---------------------- Callback ---------------------- #
def _refresh(change=None):
    fid = int(frame_slider.value)
    xyz_est, xyz_gt = _get_points_for_frame(fid)

    with fig.batch_update():
        fig.data[0].x = xyz_est[:, 0].tolist()
        fig.data[0].y = xyz_est[:, 1].tolist()
        fig.data[0].z = xyz_est[:, 2].tolist()

        fig.data[1].x = xyz_gt[:, 0].tolist()
        fig.data[1].y = xyz_gt[:, 1].tolist()
        fig.data[1].z = xyz_gt[:, 2].tolist()

        fig.layout.title = f"All keypoints - est vs GT (frame={fid})"

        # Update axis ranges dynamically if not fixed
        if ("FIX_AXIS_RANGES" not in globals()) or (not FIX_AXIS_RANGES):
            ranges = _axis_ranges(xyz_est, xyz_gt)
            if ranges is not None:
                (xr, yr, zr) = ranges
                fig.layout.scene.xaxis.range = list(xr)
                fig.layout.scene.yaxis.range = list(yr)
                fig.layout.scene.zaxis.range = list(zr)


frame_slider.observe(_refresh, names="value")
plot_in_selected_frame.observe(_refresh, names="value")

display(widgets.HBox([frame_slider, plot_in_selected_frame]))
display(fig)




### Global optimized landmarks vs GT landmark positions (keyframe slider)

This block overlays the **globally-optimized landmarks** (from `obj_key_points` snapshots at keyframes) with a **GT landmark position** estimate in the **object frame**.

GT landmark positions are computed by:
- taking each frame’s observed 3D points `reg_curr3d` (camera frame),
- transforming them into the object frame using the **GT pose** (`T_c2o = inv(T_o2c_gt)`),
- aggregating per-landmark using `reg_key_points_idx` (landmark index) and averaging.

If GT poses or required log fields are missing, the GT overlay will be unavailable.


In [ ]:
# --- Plotly 3D: global optimized landmarks (per keyframe) vs GT landmark mean positions ---

# Requirements:
#   - Run the earlier cells that define: meta, frame_ids, kf_rows, global_landmarks_updates, _ragged_slice/_ragged_slice_2d
#   - For GT overlay: dataset reader must be available (reader != None) and meta must contain:
#       obj_kp_3d_camera_data, obj_kp_3d_camera_offsets, obj_kp_3d_camera_lengths
#       obj_key_point_frames_data, obj_key_point_frames_offsets, obj_key_point_frames_lengths

import numpy as np

LANDMARK_MARKER_SIZE = 4  # increase/decrease for visibility

try:
    import plotly.graph_objects as go
except Exception as e:
    raise ImportError("Plotly is required for this interactive plot. Install with `pip install plotly`.") from e

from point2pose.utils.transform import transform_pts, inverse_SE3

# ---------------- GT landmark mean computation ---------------- #
gt_landmarks_xyz = None
gt_landmarks_count = None

# Determine max landmark id across snapshots
if "global_landmarks_updates" not in globals() or not isinstance(global_landmarks_updates, list) or len(global_landmarks_updates) == 0:
    raise RuntimeError("global_landmarks_updates is missing/empty. Run the replay cell that builds it first.")

_updates_all = [u for u in global_landmarks_updates if int(u.get("obj_id", -1)) == int(OBJ_ID)]
if len(_updates_all) == 0:
    raise RuntimeError(f"No snapshots found for OBJ_ID={OBJ_ID}.")

updates = sorted(_updates_all, key=lambda u: int(u.get("kf_idx", -1)))

_all_ids = []
for u in updates:
    ids = u.get("ids", None)
    if ids is None:
        continue
    ids = np.asarray(ids, dtype=int).reshape(-1)
    if ids.size:
        _all_ids.append(ids)

if len(_all_ids) == 0:
    raise RuntimeError("No landmark ids found in snapshots.")

all_ids = np.concatenate(_all_ids, axis=0)
max_id = int(np.max(all_ids))
print(f"[GT overlay] Max landmark id across snapshots: {max_id}")

# Check prerequisites for GT
_has_reader = ("reader" in globals()) and (reader is not None)
_has_pose_getter = ("_reader_get_gt_pose" in globals()) and callable(_reader_get_gt_pose)
_has_kp_camera = "obj_kp_3d_camera_data" in meta.files
_has_key_point_frames = "obj_key_point_frames_data" in meta.files

if (not _has_reader) or (not _has_pose_getter) or (not _has_kp_camera) or (not _has_key_point_frames):
    print(
        "[GT overlay] Skipping GT landmark computation: missing reader/get_gt_pose helper or required log fields "
        "(obj_kp_3d_camera, obj_key_point_frames)."
    )
    gt_landmarks_xyz = np.full((max_id + 1, 3), np.nan, dtype=float)
    gt_landmarks_count = np.zeros((max_id + 1,), dtype=int)
else:
    # Accumulate GT object-frame positions per landmark id using camera frame keypoints from keyframes
    sum_xyz = np.zeros((max_id + 1, 3), dtype=float)
    cnt = np.zeros((max_id + 1,), dtype=int)

    gt_pose_0 = _reader_get_gt_pose(0)
    if gt_pose_0 is None:
        for fid in range(len(reader)):
            gt_pose_0 = _reader_get_gt_pose(fid)
            if gt_pose_0 is not None:
                print(f"[GT overlay] Using frame {fid} as GT reference for conversion.")
                break

    if gt_pose_0 is None:
        print("[GT overlay] No GT poses found in reader; GT overlay unavailable.")
        gt_landmarks_xyz = np.full((max_id + 1, 3), np.nan, dtype=float)
        gt_landmarks_count = np.zeros((max_id + 1,), dtype=int)
    else:
        gt_pose_0 = np.asarray(gt_pose_0, dtype=float)

        used_keyframes = 0
        # Process only keyframes
        for kf_idx, row_idx in enumerate(kf_rows.tolist()):
            fid = int(frame_ids[row_idx])
            if fid < 0 or fid >= len(reader):
                continue

            gt_pose = _reader_get_gt_pose(fid)
            if gt_pose is None:
                continue
            gt_pose = np.asarray(gt_pose, dtype=float)

            # Get newly initialized keypoints in camera frame
            xyz_kp_camera = _ragged_slice_2d(meta, "obj_kp_3d_camera", row_idx, 3, dtype=float)
            if xyz_kp_camera.shape[0] == 0:
                continue

            # Get key point frames to identify which keypoints were initialized at this keyframe
            key_point_frames = _ragged_slice(meta, "obj_key_point_frames", row_idx).astype(int, copy=False).reshape(-1)
            if key_point_frames.size == 0:
                continue

            # Get all keypoints to find indices
            xyz_key_points = _ragged_slice_2d(meta, "obj_key_points", row_idx, 3, dtype=float)
            if xyz_key_points.shape[0] == 0:
                continue

            # Only use key points that were initialized at this keyframe
            # The camera frame keypoints correspond to newly initialized ones
            # We need to match them with key_point_frames
            mask_init_here = key_point_frames == fid
            if not np.any(mask_init_here):
                continue

            # The obj_kp_3d_camera contains only the newly initialized keypoints for this keyframe
            # So we can use all of them (they are already filtered to this keyframe)
            xyz_kf_camera = xyz_kp_camera

            # Get landmark IDs for the newly initialized keypoints
            # Find the indices in the full keypoint array that correspond to this keyframe
            ids_kf = np.flatnonzero(mask_init_here).astype(int)

            # Make sure sizes match
            if ids_kf.size != xyz_kf_camera.shape[0]:
                # If sizes do not match, use the minimum
                n = min(ids_kf.size, xyz_kf_camera.shape[0])
                ids_kf = ids_kf[:n]
                xyz_kf_camera = xyz_kf_camera[:n]
                if n == 0:
                    continue

            # Filter valid keypoints if available
            if "obj_valid_data" in meta.files:
                valid = _ragged_slice(meta, "obj_valid", row_idx).astype(bool, copy=False).reshape(-1)
                if valid.size == xyz_key_points.shape[0]:
                    valid_mask = valid[mask_init_here]
                    if valid_mask.size == xyz_kf_camera.shape[0]:
                        xyz_kf_camera = xyz_kf_camera[valid_mask]
                        ids_kf = ids_kf[valid_mask]
                        if xyz_kf_camera.shape[0] == 0:
                            continue

            # Filter finite points
            m = np.isfinite(xyz_kf_camera).all(axis=1) & (ids_kf >= 0) & (ids_kf <= max_id)
            if not np.any(m):
                continue

            xyz_kf_camera = xyz_kf_camera[m]
            ids_kf = ids_kf[m]

            # Transform keypoints from camera frame to object frame using GT pose
            # gt_pose is object->camera, so inverse(gt_pose) transforms camera->object
            T_c2o_gt = inverse_SE3(gt_pose @ inverse_SE3(gt_pose_0))
            xyz_obj = transform_pts(T_c2o_gt, xyz_kf_camera)
            ids_m = ids_kf

            np.add.at(sum_xyz, ids_m, xyz_obj)
            np.add.at(cnt, ids_m, 1)
            used_keyframes += 1

        gt_landmarks_xyz = np.full((max_id + 1, 3), np.nan, dtype=float)
        ok = cnt > 0
        gt_landmarks_xyz[ok] = sum_xyz[ok] / cnt[ok, None]
        gt_landmarks_count = cnt

        print(f"[GT overlay] Computed GT mean positions for {int(np.sum(ok))} landmarks using {used_keyframes} keyframes.")

# ---------------- Plotly slider over keyframe snapshots ---------------- #
# Precompute axis ranges using EST + (available) GT
_xyz_all = []
for u in updates:
    xyz = np.asarray(u["xyz"], dtype=float)
    m = np.isfinite(xyz).all(axis=1)
    if np.any(m):
        _xyz_all.append(xyz[m])

    ids = np.asarray(u["ids"], dtype=int).reshape(-1)
    if gt_landmarks_xyz is not None and ids.size:
        gt = gt_landmarks_xyz[ids]
        mg = np.isfinite(gt).all(axis=1)
        if np.any(mg):
            _xyz_all.append(gt[mg])

_xyz_all = np.concatenate(_xyz_all, axis=0) if len(_xyz_all) else np.zeros((0, 3), dtype=float)
if _xyz_all.shape[0] == 0:
    raise RuntimeError("No finite points found for plotting.")

xmin, ymin, zmin = np.min(_xyz_all, axis=0)
xmax, ymax, zmax = np.max(_xyz_all, axis=0)

# Axis-equal (cube) ranges
spans = np.array([xmax - xmin, ymax - ymin, zmax - zmin], dtype=float)
spans = np.maximum(spans, 1e-6)
max_span = float(np.max(spans))

# Pad relative to the max span so all axes stay equal
pad = 0.05 * max_span
half = 0.5 * max_span + pad

cx, cy, cz = float(0.5 * (xmin + xmax)), float(0.5 * (ymin + ymax)), float(0.5 * (zmin + zmax))

xr = [cx - half, cx + half]
yr = [cy - half, cy + half]
zr = [cz - half, cz + half]


def make_frame(u):
    xyz_est = np.asarray(u["xyz"], dtype=float)
    ids = np.asarray(u["ids"], dtype=int).reshape(-1)

    # GT for matching ids (may be NaN / missing)
    gt = gt_landmarks_xyz[ids] if gt_landmarks_xyz is not None else np.full_like(xyz_est, np.nan)

    m_est = np.isfinite(xyz_est).all(axis=1)
    m_gt = np.isfinite(gt).all(axis=1)

    # Only show points that exist in EST; GT shown only where available for those ids
    xyz_est_f = xyz_est[m_est]
    ids_f = ids[m_est]

    gt_f = gt[m_est]
    m_gt_f = np.isfinite(gt_f).all(axis=1)

    traces = []

    traces.append(
        go.Scatter3d(
            x=xyz_est_f[:, 0], y=xyz_est_f[:, 1], z=xyz_est_f[:, 2],
            mode="markers",
            name="Optimized landmarks (est)",
            marker=dict(size=LANDMARK_MARKER_SIZE, color="rgba(0, 114, 178, 0.85)"),
            text=[f"id={int(i)}" for i in ids_f],
            hoverinfo="text",
        )
    )

    if np.any(m_gt_f):
        traces.append(
            go.Scatter3d(
                x=gt_f[m_gt_f, 0], y=gt_f[m_gt_f, 1], z=gt_f[m_gt_f, 2],
                mode="markers",
                name="GT landmark mean",
                marker=dict(size=LANDMARK_MARKER_SIZE, color="rgba(213, 94, 0, 0.85)"),
                text=[f"id={int(i)} (n={int(gt_landmarks_count[int(i)])})" for i in ids_f[m_gt_f]],
                hoverinfo="text",
            )
        )
    else:
        # Keep legend stable
        traces.append(
            go.Scatter3d(
                x=[],
                y=[],
                z=[],
                mode="markers",
                name="GT landmark mean",
                marker=dict(size=LANDMARK_MARKER_SIZE, color="rgba(213, 94, 0, 0.85)"),
            )
        )

    title = f"kf_idx={int(u.get('kf_idx', -1))} | frame_id={int(u.get('frame_id', -1))} | est={xyz_est_f.shape[0]} | gt_avail={int(np.sum(m_gt_f))}"
    return title, traces


# Build frames
frames = []
titles = []
for u in updates:
    title, traces = make_frame(u)
    titles.append(title)
    # Use frame_id for frame name instead of kf_idx
    frame_id = int(u.get("frame_id", -1))
    frames.append(go.Frame(data=traces, name=str(frame_id), layout=go.Layout(title=title)))

# Initial frame
init_traces = frames[0].data
init_title = titles[0]

steps = []
for i, u in enumerate(updates):
    # Use frame_id for slider instead of kf_idx
    frame_id = int(u.get("frame_id", -1))
    fname = str(frame_id)
    steps.append(
        dict(
            method="animate",
            args=[[fname], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}],
            label=fname,
        )
    )

sliders = [dict(active=0, currentvalue={"prefix": "frame_id: "}, pad={"t": 50}, steps=steps)]

# Get plot dimensions (make it bigger)
plot_width = PLOT_WIDTH if "PLOT_WIDTH" in globals() else 1600
plot_height = PLOT_HEIGHT if "PLOT_HEIGHT" in globals() else 1000

fig = go.Figure(
    data=init_traces,
    layout=go.Layout(
        title=init_title,
        width=plot_width,
        height=plot_height,
        paper_bgcolor="white",
        plot_bgcolor="white",
        scene=dict(
            xaxis=dict(range=xr, visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
            yaxis=dict(range=yr, visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
            zaxis=dict(range=zr, visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
            bgcolor="white",
            aspectmode="cube",  # axis-equal
        ),
        legend=dict(x=0.01, y=0.99),
        sliders=sliders,
        updatemenus=[
            dict(
                type="buttons",
                showactive=False,
                x=0.01,
                y=0.0,
                xanchor="left",
                yanchor="bottom",
                pad=dict(t=45, r=10),
                buttons=[
                    dict(label="Play", method="animate", args=[None, {"frame": {"duration": 200, "redraw": True}, "transition": {"duration": 0}}]),
                    dict(label="Pause", method="animate", args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]),
                ],
            )
        ],
    ),
    frames=frames,
)

fig.show()





### Global optimized landmarks colored by height (z)


In [ ]:
# --- Plotly 3D: global optimized landmarks colored by height (z) ---

# Requirements:
#   - Run the earlier cells that define: global_landmarks_updates, OBJ_ID

try:
    import plotly.graph_objects as go
except Exception as e:
    raise ImportError("Plotly is required for this interactive plot. Install with `pip install plotly`.") from e

if "global_landmarks_updates" not in globals() or not isinstance(global_landmarks_updates, list):
    print("No global_landmarks_updates found. Run the replay cell above first.")
elif len(global_landmarks_updates) == 0:
    print("global_landmarks_updates is empty.")
else:
    updates = [u for u in global_landmarks_updates if int(u.get("obj_id", -1)) == int(OBJ_ID)]
    if len(updates) == 0:
        print(f"No landmark snapshots found for OBJ_ID={OBJ_ID}.")
    else:
        updates = sorted(updates, key=lambda u: int(u.get("kf_idx", -1)))

        all_xyz = []
        all_z = []
        for u in updates:
            xyz = np.asarray(u.get("xyz", np.zeros((0, 3))), dtype=float).reshape(-1, 3)
            m = np.isfinite(xyz).all(axis=1)
            if np.any(m):
                xyz = xyz[m]
                all_xyz.append(xyz)
                all_z.append(xyz[:, 2])

        if len(all_xyz) == 0:
            raise RuntimeError("No finite landmark xyz found for height-color plot.")

        xyz_all = np.concatenate(all_xyz, axis=0)
        z_all = np.concatenate(all_z, axis=0)

        zmin = float(np.min(z_all))
        zmax = float(np.max(z_all))

        xmin, ymin, zmin_axis = np.min(xyz_all, axis=0)
        xmax, ymax, zmax_axis = np.max(xyz_all, axis=0)

        spans = np.maximum(np.array([xmax - xmin, ymax - ymin, zmax_axis - zmin_axis], dtype=float), 1e-6)
        max_span = float(np.max(spans))
        pad = 0.05 * max_span
        half = 0.5 * max_span + pad

        cx, cy, cz = 0.5 * (xmin + xmax), 0.5 * (ymin + ymax), 0.5 * (zmin_axis + zmax_axis)
        xr = [float(cx - half), float(cx + half)]
        yr = [float(cy - half), float(cy + half)]
        zr = [float(cz - half), float(cz + half)]

        fig = go.Figure()

        for k, u in enumerate(updates):
            xyz = np.asarray(u.get("xyz", np.zeros((0, 3))), dtype=float).reshape(-1, 3)
            ids = np.asarray(u.get("ids", np.zeros((0,), dtype=int)), dtype=int).reshape(-1)
            m = np.isfinite(xyz).all(axis=1)
            xyz = xyz[m]
            ids = ids[m] if ids.size == m.size else np.arange(xyz.shape[0], dtype=int)

            frame_id = int(u.get("frame_id", -1))
            kf_idx = int(u.get("kf_idx", -1))

            fig.add_trace(
                go.Scatter3d(
                    x=xyz[:, 0],
                    y=xyz[:, 1],
                    z=xyz[:, 2],
                    mode="markers",
                    marker=dict(
                        size=5,
                        color=xyz[:, 2],
                        colorscale="Viridis",
                        cmin=zmin,
                        cmax=zmax,
                        showscale=True,
                        colorbar=dict(title="height (z)"),
                    ),
                    text=[f"id={int(i)}" for i in ids],
                    hovertemplate="x=%{x:.4f}<br>y=%{y:.4f}<br>z=%{z:.4f}<br>%{text}<extra></extra>",
                    name=f"kf_idx={kf_idx}, frame_id={frame_id}",
                    visible=(k == 0),
                )
            )

        steps = []
        for k, u in enumerate(updates):
            vis = [False] * len(updates)
            vis[k] = True
            frame_id = int(u.get("frame_id", -1))
            kf_idx = int(u.get("kf_idx", -1))
            steps.append(
                dict(
                    method="update",
                    args=[
                        {"visible": vis},
                        {"title": f"Landmarks colored by height (obj {OBJ_ID}) - kf_idx={kf_idx}, frame_id={frame_id}"},
                    ],
                    label=str(frame_id if frame_id >= 0 else kf_idx),
                )
            )

        fig.update_layout(
            title=f"Landmarks colored by height (obj {OBJ_ID}) - kf_idx={int(updates[0].get('kf_idx', -1))}, frame_id={int(updates[0].get('frame_id', -1))}",
            width=PLOT_WIDTH if "PLOT_WIDTH" in globals() else 1200,
            height=PLOT_HEIGHT if "PLOT_HEIGHT" in globals() else 850,
            paper_bgcolor="white",
            plot_bgcolor="white",
            margin=dict(l=0, r=0, b=0, t=55),
            scene=dict(
                xaxis=dict(range=xr, visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
                yaxis=dict(range=yr, visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
                zaxis=dict(range=zr, visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
                bgcolor="white",
                aspectmode="cube",
            ),
            sliders=[dict(active=0, currentvalue={"prefix": "frame_id: "}, steps=steps)],
            legend=dict(x=0.01, y=0.99),
            uirevision=f"kf_landmarks_height_obj_{OBJ_ID}",
        )

        fig.show()


### Global optimized landmarks with custom coloring

Two additional figures are shown below:
- Uniform color for all landmarks.
- Color by the keyframe index where each landmark was first added.

Both use larger marker size for better visibility.


In [ ]:
# --- Plotly 3D: global optimized landmarks with uniform color + init keyframe color ---

import numpy as np

try:
    import plotly.graph_objects as go
except Exception as e:
    raise ImportError("Plotly is required for this interactive plot. Install with `pip install plotly`.") from e

LARGE_MARKER_SIZE = 8

if "global_landmarks_updates" not in globals() or not isinstance(global_landmarks_updates, list):
    print("No global_landmarks_updates found. Run the replay cell above first.")
elif len(global_landmarks_updates) == 0:
    print("global_landmarks_updates is empty.")
else:
    updates = [u for u in global_landmarks_updates if int(u.get("obj_id", -1)) == int(OBJ_ID)]
    if len(updates) == 0:
        print(f"No landmark snapshots found for OBJ_ID={OBJ_ID}.")
    else:
        updates = sorted(updates, key=lambda u: int(u.get("kf_idx", -1)))

        all_xyz = []
        for u in updates:
            xyz = np.asarray(u.get("xyz", np.zeros((0, 3))), dtype=float).reshape(-1, 3)
            m = np.isfinite(xyz).all(axis=1)
            if np.any(m):
                all_xyz.append(xyz[m])

        if len(all_xyz) == 0:
            raise RuntimeError("No finite landmark xyz found for plotting.")

        xyz_all = np.concatenate(all_xyz, axis=0)
        xmin, ymin, zmin = np.min(xyz_all, axis=0)
        xmax, ymax, zmax = np.max(xyz_all, axis=0)

        spans = np.maximum(np.array([xmax - xmin, ymax - ymin, zmax - zmin], dtype=float), 1e-6)
        max_span = float(np.max(spans))
        pad = 0.05 * max_span
        half = 0.5 * max_span + pad

        cx, cy, cz = 0.5 * (xmin + xmax), 0.5 * (ymin + ymax), 0.5 * (zmin + zmax)
        xr = [float(cx - half), float(cx + half)]
        yr = [float(cy - half), float(cy + half)]
        zr = [float(cz - half), float(cz + half)]

        # Track first keyframe index where each landmark id appears.
        landmark_first_kf = {}
        for u in updates:
            kf_idx = int(u.get("kf_idx", -1))
            ids = np.asarray(u.get("ids", np.zeros((0,), dtype=int)), dtype=int).reshape(-1)
            for lid in np.unique(ids):
                lid_i = int(lid)
                if lid_i < 0:
                    continue
                if lid_i not in landmark_first_kf:
                    landmark_first_kf[lid_i] = kf_idx

        if len(landmark_first_kf) > 0:
            init_kf_min = int(min(landmark_first_kf.values()))
            init_kf_max = int(max(landmark_first_kf.values()))
        else:
            init_kf_min, init_kf_max = 0, 1

        if init_kf_min == init_kf_max:
            init_kf_max = init_kf_min + 1

        def _extract_xyz_ids(update):
            xyz = np.asarray(update.get("xyz", np.zeros((0, 3))), dtype=float).reshape(-1, 3)
            ids = np.asarray(update.get("ids", np.zeros((0,), dtype=int)), dtype=int).reshape(-1)

            m = np.isfinite(xyz).all(axis=1)
            xyz = xyz[m]
            if ids.size == m.size:
                ids = ids[m]
            else:
                ids = np.arange(xyz.shape[0], dtype=int)
            return xyz, ids

        # ---------------- Figure 1: uniform color ---------------- #
        fig_uniform = go.Figure()

        for k, u in enumerate(updates):
            xyz, ids = _extract_xyz_ids(u)
            frame_id = int(u.get("frame_id", -1))
            kf_idx = int(u.get("kf_idx", -1))

            fig_uniform.add_trace(
                go.Scatter3d(
                    x=xyz[:, 0],
                    y=xyz[:, 1],
                    z=xyz[:, 2],
                    mode="markers",
                    marker=dict(size=LARGE_MARKER_SIZE, color="rgba(0, 114, 178, 0.92)"),
                    text=[f"id={int(i)}" for i in ids],
                    hovertemplate="x=%{x:.4f}<br>y=%{y:.4f}<br>z=%{z:.4f}<br>%{text}<extra></extra>",
                    name=f"kf_idx={kf_idx}, frame_id={frame_id}",
                    visible=(k == 0),
                )
            )

        uniform_steps = []
        for k, u in enumerate(updates):
            vis = [False] * len(updates)
            vis[k] = True
            frame_id = int(u.get("frame_id", -1))
            kf_idx = int(u.get("kf_idx", -1))
            uniform_steps.append(
                dict(
                    method="update",
                    args=[
                        {"visible": vis},
                        {
                            "title": (
                                f"Landmarks (uniform color, obj {OBJ_ID}) - "
                                f"kf_idx={kf_idx}, frame_id={frame_id}"
                            )
                        },
                    ],
                    label=str(frame_id if frame_id >= 0 else kf_idx),
                )
            )

        fig_uniform.update_layout(
            title=(
                f"Landmarks (uniform color, obj {OBJ_ID}) - "
                f"kf_idx={int(updates[0].get('kf_idx', -1))}, frame_id={int(updates[0].get('frame_id', -1))}"
            ),
            width=PLOT_WIDTH if "PLOT_WIDTH" in globals() else 1200,
            height=PLOT_HEIGHT if "PLOT_HEIGHT" in globals() else 850,
            paper_bgcolor="white",
            plot_bgcolor="white",
            margin=dict(l=0, r=0, b=0, t=55),
            scene=dict(
                xaxis=dict(range=xr, visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
                yaxis=dict(range=yr, visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
                zaxis=dict(range=zr, visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
                bgcolor="white",
                aspectmode="cube",
            ),
            sliders=[dict(active=0, currentvalue={"prefix": "frame_id: "}, steps=uniform_steps)],
            legend=dict(x=0.01, y=0.99),
            uirevision=f"kf_landmarks_uniform_obj_{OBJ_ID}",
        )

        fig_uniform.show()

        # ---------------- Figure 2: color by landmark init keyframe ---------------- #
        fig_init_kf = go.Figure()

        for k, u in enumerate(updates):
            xyz, ids = _extract_xyz_ids(u)
            frame_id = int(u.get("frame_id", -1))
            kf_idx = int(u.get("kf_idx", -1))

            init_kf = np.asarray([landmark_first_kf.get(int(i), kf_idx) for i in ids], dtype=float)

            fig_init_kf.add_trace(
                go.Scatter3d(
                    x=xyz[:, 0],
                    y=xyz[:, 1],
                    z=xyz[:, 2],
                    mode="markers",
                    marker=dict(
                        size=LARGE_MARKER_SIZE,
                        color=init_kf,
                        colorscale="Turbo",
                        cmin=init_kf_min,
                        cmax=init_kf_max,
                        showscale=True,
                        colorbar=dict(title="init kf_idx"),
                    ),
                    text=[
                        f"id={int(i)} | init_kf_idx={int(ik)}"
                        for i, ik in zip(ids, init_kf)
                    ],
                    hovertemplate="x=%{x:.4f}<br>y=%{y:.4f}<br>z=%{z:.4f}<br>%{text}<extra></extra>",
                    name=f"kf_idx={kf_idx}, frame_id={frame_id}",
                    visible=(k == 0),
                )
            )

        init_kf_steps = []
        for k, u in enumerate(updates):
            vis = [False] * len(updates)
            vis[k] = True
            frame_id = int(u.get("frame_id", -1))
            kf_idx = int(u.get("kf_idx", -1))
            init_kf_steps.append(
                dict(
                    method="update",
                    args=[
                        {"visible": vis},
                        {
                            "title": (
                                f"Landmarks colored by init keyframe (obj {OBJ_ID}) - "
                                f"kf_idx={kf_idx}, frame_id={frame_id}"
                            )
                        },
                    ],
                    label=str(frame_id if frame_id >= 0 else kf_idx),
                )
            )

        fig_init_kf.update_layout(
            title=(
                f"Landmarks colored by init keyframe (obj {OBJ_ID}) - "
                f"kf_idx={int(updates[0].get('kf_idx', -1))}, frame_id={int(updates[0].get('frame_id', -1))}"
            ),
            width=PLOT_WIDTH if "PLOT_WIDTH" in globals() else 1200,
            height=PLOT_HEIGHT if "PLOT_HEIGHT" in globals() else 850,
            paper_bgcolor="white",
            plot_bgcolor="white",
            margin=dict(l=0, r=0, b=0, t=55),
            scene=dict(
                xaxis=dict(range=xr, visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
                yaxis=dict(range=yr, visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
                zaxis=dict(range=zr, visible=False, showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
                bgcolor="white",
                aspectmode="cube",
            ),
            sliders=[dict(active=0, currentvalue={"prefix": "frame_id: "}, steps=init_kf_steps)],
            legend=dict(x=0.01, y=0.99),
            uirevision=f"kf_landmarks_init_kf_obj_{OBJ_ID}",
        )

        fig_init_kf.show()
